# Step 1: Create PR Table
- This notebook uses SQL queries to generate a list of provider metrics for each opioid prescriber with a pharmacy claim in the time frame listed
- Step1: Fill in begin and end dates
- Step2: Fill in the connection information, this will be the oracle DB connection with your credentials
- Step3: Fill in the file prefix for the resultant file naming and if you set save_csv to "True" it will create a csv file in the data assets filder

## 1. Setup Notebook Inputs
- Begin and End Dates for query timeline
- UAT connection credentials for access to the database
- path is where the data assets will be saved/read from
- file prefix for the naming of the CSV file. The final file will be file_prefix_begin_date_end_date i.e op_analysis_01012020_03312020.csv
- save_csv is the option to save the analysis file. When testing set to False

In [1]:
begin_date = '04012020'
end_date = '06302020'
uat_connection = 'alex_uat'
path = '/project_data/data_asset'
file_prefix = 'pipeline_op_analysis'
save_csv = True

## Import libraries

In [2]:
import pandas as pd
import cx_Oracle
from sqlalchemy import create_engine, types
pd.set_option('display.max_columns', None)
from scipy import stats
import json
import re

#arcgis
# !pip install arcgis
# from copy import deepcopy
# from datetime import datetime
# from IPython.display import HTML
# from arcgis.gis import GIS
# import arcgis.network as network
# import arcgis.geocoding as geocoding

## Get credentials for connection

In [3]:
# @hidden_cell
# The following code contains the credentials for a connection in your Project.
# You might want to remove those credentials before you share your notebook.

from project_lib import Project
project = Project.access()
uat_credentials = project.get_connection(name=uat_connection)


# api_creds = project.get_file(file_name = "creds.json")

# with open('creds.json', "wb") as outfile:
#         outfile.write(api_creds.getbuffer())

# with open('creds.json') as f:
#   creds = json.load(f)

## Setup DB Connection

In [4]:
host = uat_credentials['host']
port = uat_credentials['port']
user = uat_credentials['username']
password = uat_credentials['password']
service_name = uat_credentials['service_name']

sid = cx_Oracle.makedsn(host = host, 
                        port = port, 
                        service_name = service_name)
 
cstr = 'oracle://{user}:{password}@{sid}'.format(
    user=user,
    password=password,
    sid=sid
)

engine =  create_engine(
    cstr,
    convert_unicode=False,
    pool_recycle=10,
    pool_size=50,
    echo=True
)

print(user)

u4j8175


## Function to drop table if exists

In [5]:
def drop_table_if_exists(table_name):
    qry = """
        BEGIN
           EXECUTE IMMEDIATE 'DROP TABLE ' || \'{tbl}\';
        EXCEPTION
           WHEN OTHERS THEN
              IF SQLCODE != -942 THEN
                 RAISE;
              END IF;
        END;    
    """.format(tbl = table_name)
    res = engine.execute(qry)
    return('ran')
    

In [6]:
drop_table_if_exists('CDC_WKBK_2018')

2020-10-28 23:05:36,425 INFO sqlalchemy.engine.base.Engine SELECT USER FROM DUAL
2020-10-28 23:05:36,428 INFO sqlalchemy.engine.base.Engine {}
2020-10-28 23:05:36,474 INFO sqlalchemy.engine.base.Engine SELECT CAST('test plain returns' AS VARCHAR(60 CHAR)) AS anon_1 FROM DUAL
2020-10-28 23:05:36,476 INFO sqlalchemy.engine.base.Engine {}
2020-10-28 23:05:36,519 INFO sqlalchemy.engine.base.Engine SELECT CAST('test unicode returns' AS NVARCHAR2(60)) AS anon_1 FROM DUAL
2020-10-28 23:05:36,521 INFO sqlalchemy.engine.base.Engine {}
2020-10-28 23:05:36,605 INFO sqlalchemy.engine.base.Engine select value from nls_session_parameters where parameter = 'NLS_NUMERIC_CHARACTERS'
2020-10-28 23:05:36,607 INFO sqlalchemy.engine.base.Engine {}
2020-10-28 23:05:36,653 INFO sqlalchemy.engine.base.Engine 
        BEGIN
           EXECUTE IMMEDIATE 'DROP TABLE ' || 'CDC_WKBK_2018';
        EXCEPTION
           WHEN OTHERS THEN
              IF SQLCODE != -942 THEN
                 RAISE;
              END 

'ran'

# Test connection

In [7]:
result_df = pd.read_sql('select * from BIDM_USR_RPTS.CLM_LNE_FACT_V where rownum <= 10', engine)

result_df.head()

2020-10-28 23:05:36,850 INFO sqlalchemy.engine.base.Engine SELECT table_name FROM all_tables WHERE table_name = :name AND owner = :schema_name
2020-10-28 23:05:36,851 INFO sqlalchemy.engine.base.Engine {'name': 'select * from BIDM_USR_RPTS.CLM_LNE_FACT_V where rownum <= 10', 'schema_name': 'U4J8175'}
2020-10-28 23:05:36,935 INFO sqlalchemy.engine.base.Engine select * from BIDM_USR_RPTS.CLM_LNE_FACT_V where rownum <= 10
2020-10-28 23:05:36,937 INFO sqlalchemy.engine.base.Engine {}


,clm_lne_fact_sk,clm_pd_dt,clm_dim_sk,icn_nbr,lne_nbr,clnt_dim_sk,mcaid_id,mcaid_id_mskd,clnt_brth_dt,clnt_age_qty,sbmt_mcaid_id,sbmt_mcaid_id_mskd,sbmt_clnt_lst_nm,sbmt_clnt_frst_nm,sbmt_clnt_mdl_nm,sbmt_clnt_sfx_nm,sbmt_clnt_titl_nm,sbmt_clnt_brth_dt,sbmt_clnt_age_qty,sbmt_clnt_gndr_cd,sbmt_clnt_gndr_desc,sbmt_prgncy_ind,clnt_cnty_geo_dim_sk,clnt_cnty_cd,clnt_cnty_desc,bill_prov_loc_dim_sk,bill_prov_loc_id,bill_prov_mcaid_id,bill_prov_npi_id,bill_prov_loc_nm,bill_prov_typ_cd,bill_prov_typ_desc,bill_prov_spclty_cd,bill_prov_spclty_desc,mc_prov_loc_dim_sk,mc_prov_loc_id,mc_prov_mcaid_id,mc_prov_npi_id,mc_prov_loc_nm,mc_prov_typ_cd,mc_prov_typ_desc,mc_prov_spclty_cd,mc_prov_spclty_desc,rend_prov_loc_dim_sk,rend_prov_loc_id,rend_prov_mcaid_id,rend_prov_npi_id,rend_prov_loc_nm,rend_prov_typ_cd,rend_prov_typ_desc,rend_prov_spclty_cd,rend_prov_spclty_desc,attd_prov_loc_dim_sk,attd_prov_loc_id,attd_prov_loc_nm,rfr_prov_loc_dim_sk,rfr_prov_loc_id,rfr_prov_loc_nm,rfr_2_prov_loc_dim_sk,rfr_2_prov_loc_id,rfr_2_prov_loc_nm,supv_prov_loc_dim_sk,supv_prov_loc_id,supv_prov_loc_nm,fac_prov_loc_dim_sk,fac_prov_loc_id,fac_prov_loc_nm,prscrb_prov_loc_dim_sk,prscrb_prov_loc_id,prscrb_prov_loc_nm,clm_ctg_cd,clm_ctg_desc,clm_typ_cd,clm_typ_desc,clm_trnsct_typ_cd,clm_trnsct_typ_desc,bill_typ_cd,bill_typ_desc,prov_svc_cnty_geo_dim_sk,prov_svc_cnty_desc,prov_svc_cnty_cd,clm_sts_cd,clm_sts_desc,clm_pmt_dt,nbr_of_lines_ct,clm_frst_svc_dt,clm_lst_svc_dt,sbmt_dt,adjdc_dt,clm_fin_prcs_dt,admsn_dt,admsn_tm,dschrg_dt,dschrg_tm,acdnt_dt,ptnt_acct_nbr,ptnt_sts_cd,ptnt_sts_desc,clm_copay_prcs_cd,clm_copay_prcs_desc,clm_bill_amt,clm_net_bill_amt,clm_tpl_pd_amt,clm_allw_amt,clm_pd_amt,clm_copay_amt,clm_ptnt_pd_amt,pos_cd,pos_desc,clm_carr_deny_ind,adj_ind,certn_cd,mrn_cd,enc_sts_cd,clm_mc_enc_pd_dt,clm_mc_enc_pd_amt,clm_mc_enc_copay_amt,mcare_dsclmr_cd,otlr_day_amt,otlr_cst_amt,adj_eob_cd,adj_eob_desc,adj_rsn_cd,adj_rsn_desc,apr_drg_cd,apr_drg_desc,admt_src_cd,admt_src_desc,enc_ind,cvr_day_qty,non_cvr_day_qty,admt_typ_cd,admt_typ_desc,rx_nbr,prsrb_dt,nrs_home_ind,drug_dspn_dt,dur_intrvn_cd,dur_otcm_cd,ptnt_loc_cd,rmbrs_basis_cd,othr_pyr_appl_amt,uc_chrg_amt,ingrd_cst_amt,dspn_fee_amt,rfl_qty,clm_dspn_unt_qty,drug_day_sply_qty,drug_brnd_nm_ind,lne_sts_cd,lne_sts_desc,lne_frst_svc_dt,lne_lst_svc_dt,orgn_icn_nbr,orgn_clm_pd_dt,orgn_pmt_dt,adj_seq_nbr,parnt_icn_nbr,parnt_clm_pd_dt,parnt_pmt_dt,most_rcnt_clm_ind,clm_adj_dt,adj_icn_nbr,adj_clm_pd_dt,adj_pmt_dt,rvrsl_ind,clm_ct,clm_lne_ct,ncpdp_othr_cvrg_cd,ncpdp_othr_cvrg_desc,phrmcs_rsn_for_svc_1_cd,phrmcs_rsn_for_svc_1_desc,phrmcs_rsn_for_svc_2_cd,phrmcs_rsn_for_svc_2_desc,phrmcs_rsn_for_svc_3_cd,phrmcs_rsn_for_svc_3_desc,phrmcs_rsn_for_svc_4_cd,phrmcs_rsn_for_svc_4_desc,drug_daw_cd,drug_daw_desc,disprt_shr_amt,drg_dly_rt_amt,drg_wght_qty,prov_loc_dly_rt_amt,pd_amt,allw_amt,copay_amt,tpl_pd_amt,bill_amt,bill_unt_qty,allw_unt_qty,non_cvr_amt,mc_enc_pd_dt,mc_enc_pd_amt,mc_enc_copay_amt,diag_1_seq_cd,diag_1_dim_sk,diag_1_cd_set_cd,diag_1_cd_set_ver_nbr,diag_1_cd,diag_1_desc,diag_2_seq_cd,diag_2_dim_sk,diag_2_cd_set_cd,diag_2_cd_set_ver_nbr,diag_2_cd,diag_2_desc,diag_3_seq_cd,diag_3_dim_sk,diag_3_cd_set_cd,diag_3_cd_set_ver_nbr,diag_3_cd,diag_3_desc,diag_4_seq_cd,diag_4_dim_sk,diag_4_cd_set_cd,diag_4_cd_set_ver_nbr,diag_4_cd,diag_4_desc,proc_dim_sk,proc_cd,proc_cd_set_cd,proc_desc,proc_mod_1_cd,proc_mod_1_desc,proc_mod_2_cd,proc_mod_2_desc,proc_mod_3_cd,proc_mod_3_desc,proc_mod_4_cd,proc_mod_4_desc,rvn_dim_sk,rvn_cd,rvn_desc,acdnt_ind,emerg_ind,carr_deny_ind,sys_added_dtl_ind,prgncy_ind,trt_diag_cd_ind,copay_prcs_cd,copay_prcs_desc,epsdt_ada_cd,epsdt_fmly_pln_rlt_cd,drug_dim_sk,ndc_cd,ndc_desc,ndc_sts_cd,ndc_sts_desc,drug_frm_cd,drug_frm_desc,hic3_thrptc_drug_cls_cd,hic3_thrptc_drug_cls_desc,drug_typ_cd,drug_typ_desc,fdb_frmltn_id,fdb_frmltn_desc,fdb_clncl_frmltn_id,fdb_clncl_frmltn_desc,fed_dea_drug_schdl_cd,fed_dea_drug_schdl_desc,drug_prc_cls_cd,drug_prc_cls_desc,drug_strngt_tx,unt_dose_typ_cd,unt_dose_typ_desc,awp_a

# Copy CDC workbook

In [8]:
drop_table_if_exists('CDC_WKBK_2018')
engine.execute('create table CDC_WKBK_2018 as select * from D98JMOHR.CDC_WKBK_2018')

2020-10-28 23:05:38,122 INFO sqlalchemy.engine.base.Engine 
        BEGIN
           EXECUTE IMMEDIATE 'DROP TABLE ' || 'CDC_WKBK_2018';
        EXCEPTION
           WHEN OTHERS THEN
              IF SQLCODE != -942 THEN
                 RAISE;
              END IF;
        END;    
    
2020-10-28 23:05:38,124 INFO sqlalchemy.engine.base.Engine {}
2020-10-28 23:05:38,209 INFO sqlalchemy.engine.base.Engine create table CDC_WKBK_2018 as select * from D98JMOHR.CDC_WKBK_2018
2020-10-28 23:05:38,212 INFO sqlalchemy.engine.base.Engine {}
2020-10-28 23:05:38,958 INFO sqlalchemy.engine.base.Engine COMMIT


# Run CS Master Query

In [9]:
drop_table_if_exists('CS_MASTER')

master_query = """
CREATE TABLE CS_MASTER AS --- CREATING A MASTER CLAIMS LIST.  THIS WILL BE AN IMPORTANT REFERENCE TABLE.  WE CAN ADJUST IT HOWEVER WE NEED (EG MORE CS NDC'S OR A DIFFERENT DATE RANGE)
SELECT CLM.PRSCRB_PROV_LOC_ID, CLM.CLM_LNE_FACT_SK, CLM.ICN_NBR, CLM.LNE_NBR, 
CLM.MCAID_ID, CLM.PD_AMT, CLM.NDC_CD, CLM.CLM_PD_DT, CLM.LNE_FRST_SVC_DT, 
CLM.PRSRB_DT, CLM.DRUG_DSPN_DT, CLM.BILL_PROV_LOC_ID, CLM.CLNT_AGE_QTY, CLM.DRUG_DAY_SPLY_QTY,
CLM.RX_NBR, CLM.DSPN_UNT_QTY, 
CDC.NDC, CDC."NDC_Numeric", CDC.PRODNME, CDC.GENNME, CDC."Master_Form", CDC."Class", CDC."Drug", CDC."LongShortActing", CDC."DEAClassCode", CDC."Strength_Per_Unit", CDC.UOM, CDC."MME_Conversion_Factor"
FROM BIDM_USR_RPTS.CLM_LNE_FACT_V CLM
INNER JOIN D98JMOHR.CDC_WKBK_2018 CDC--- CS LIST.  CAN CHANGE TO LEFT JOIN IF MASTER IS EXPANDED
ON CLM.NDC_CD = CDC.NDC 
AND CLM.CLM_FRST_SVC_DT BETWEEN TO_DATE(\'{date1}\', 'MMDDYYYY') AND TO_DATE(\'{date2}\', 'MMDDYYYY') --- can be any dates
AND CLM.CLM_PD_DT >= TO_DATE(\'{date1}\', 'MMDDYYYY')
AND CLM.CLM_TYP_CD IN ('P','Q') --- pharmacy claims
AND CLM.PD_AMT > 0 --- we paid comething
AND CLM.CURR_REC_IND = 'Y'--- current record
AND CLM.SRC_REC_DEL_IND = 'N' --- source record deletion indicator
AND CLM.MOST_RCNT_CLM_IND = 'Y' ---- most recent paid claim
AND CLM.RVRSL_IND = 'N' ---- no reversals
AND CLM.ENC_IND = 'N' ---- no encounters
AND CLM.CLM_STS_CD = 'P' --- paid claim
AND CLM.LNE_STS_CD = 'P' --- paid line
""".format(date1 = begin_date, date2 = end_date)
engine.execute(master_query)

master_df = pd.read_sql('select * from CS_MASTER where rownum<10', engine)

master_df.head()

2020-10-28 23:05:39,055 INFO sqlalchemy.engine.base.Engine 
        BEGIN
           EXECUTE IMMEDIATE 'DROP TABLE ' || 'CS_MASTER';
        EXCEPTION
           WHEN OTHERS THEN
              IF SQLCODE != -942 THEN
                 RAISE;
              END IF;
        END;    
    
2020-10-28 23:05:39,057 INFO sqlalchemy.engine.base.Engine {}
2020-10-28 23:05:39,181 INFO sqlalchemy.engine.base.Engine 
CREATE TABLE CS_MASTER AS --- CREATING A MASTER CLAIMS LIST.  THIS WILL BE AN IMPORTANT REFERENCE TABLE.  WE CAN ADJUST IT HOWEVER WE NEED (EG MORE CS NDC'S OR A DIFFERENT DATE RANGE)
SELECT CLM.PRSCRB_PROV_LOC_ID, CLM.CLM_LNE_FACT_SK, CLM.ICN_NBR, CLM.LNE_NBR, 
CLM.MCAID_ID, CLM.PD_AMT, CLM.NDC_CD, CLM.CLM_PD_DT, CLM.LNE_FRST_SVC_DT, 
CLM.PRSRB_DT, CLM.DRUG_DSPN_DT, CLM.BILL_PROV_LOC_ID, CLM.CLNT_AGE_QTY, CLM.DRUG_DAY_SPLY_QTY,
CLM.RX_NBR, CLM.DSPN_UNT_QTY, 
CDC.NDC, CDC."NDC_Numeric", CDC.PRODNME, CDC.GENNME, CDC."Master_Form", CDC."Class", CDC."Drug", CDC."LongShortActing", CDC."DEAC

,prscrb_prov_loc_id,clm_lne_fact_sk,icn_nbr,lne_nbr,mcaid_id,pd_amt,ndc_cd,clm_pd_dt,lne_frst_svc_dt,prsrb_dt,drug_dspn_dt,bill_prov_loc_id,clnt_age_qty,drug_day_sply_qty,rx_nbr,dspn_unt_qty,ndc,NDC_Numeric,prodnme,gennme,Master_Form,Class,Drug,LongShortActing,DEAClassCode,Strength_Per_Unit,uom,MME_Conversion_Factor
0,135154,912661954,2520164039658,1,I785379,9.76,00591325601,2020-06-12,2020-06-11,2019-11-12,2020-06-11,151425,42,30.0,954054,30.0,00591325601,591325601,CYCLOBENZAPRINE HCL,Cyclobenzaprine Hydrochloride,Tablet,Muscle Relaxant,Cyclobenzaprine,None,6,5.0,MG,NaN
1,103763,912512367,2520159010283,1,P229338,12.67,00228300511,2020-06-12,2020-06-06,2020-05-05,2020-06-06,103272,39,30.0,5006051,60.0,00228300511,228300511,CLONAZEPAM,Clonazepam,Tablet,Benzo,Clonazepam,None,4,2.0,MG,NaN
2,116534,912356425,2520161034045,1,P367494,53.28,00781237101,2020-06-12,2020-06-08,2020-06-03,2020-06-08,106715,30,30.0,1082389,60.0,00781237101,781237101,MIXED AMPHETAMINE SALT,Amphetamine Salt Combination,"Capsule, Extended Release",Stimulant,Amphetamine,None,2,30.0,MG,NaN
3,136774,912502271,2520158005635,1,S510205,11.79,50458058701,2020-06-12,2020-06-05,2020-05-08,2020-06-05,129512,37,30.0,2146429,30.0,50458058701,50458058701,CONCERTA,Methylphenidate Hydrochloride,"Tablet, Extended Release",Stimulant,Methylphenidate,None,2,54.0,MG,NaN
4,103845,912506149,2520164013758,1,V735724,9.51,51862006205,2020-06-12,2020-06-11,2020-06-03,2020-06-11,115355,24,30.0,2636792,10.0,51862006205,51862006205,DIAZEPAM,Diazepam,Tablet,Benzo,Diazepam,None,4,2.0,MG,NaN


# Metric PR1 - PR4

In [10]:
drop_table_if_exists('CS_MASTER_SUM')

p1_qry ="""
CREATE TABLE CS_MASTER_SUM AS
SELECT A.*,
((A."Max Dispense Date" - A."Min Dispense Date")+1) AS "Total Days",
(A."Count Script"/ A."Member Count") AS PR2,
(A."Sum Dispensed Quantity"/ A."Member Count") AS PR1,
(A."Count Script"/((A."Max Dispense Date" - A."Min Dispense Date")+1)) AS PR4,
(A."Sum Dispensed Quantity"/((A."Max Dispense Date" - A."Min Dispense Date")+1)) AS PR3
FROM
(SELECT
COUNT(DISTINCT CLM.MCAID_ID) AS "Member Count", --- unique beneficiaries
COUNT(CLM.RX_NBR) AS "Count Script", --- how many RXs written
SUM(CLM.DSPN_UNT_QTY) AS "Sum Dispensed Quantity",
MAX(CLM.DRUG_DSPN_DT) AS "Max Dispense Date",
MIN(CLM.DRUG_DSPN_DT) AS "Min Dispense Date",
CLM.PRSCRB_PROV_LOC_ID
FROM CS_MASTER CLM
INNER JOIN D98JMOHR.CDC_WKBK_2018 CDC
ON CLM.NDC_CD = CDC.NDC --- CS LIST
GROUP BY 
  CLM.PRSCRB_PROV_LOC_ID) A
""".format(date1 = begin_date, date2 = end_date)

engine.execute(p1_qry)


p1_df = pd.read_sql('select * from CS_MASTER_SUM where rownum < 10', engine)

p1_df.head()

2020-10-28 23:05:44,806 INFO sqlalchemy.engine.base.Engine 
        BEGIN
           EXECUTE IMMEDIATE 'DROP TABLE ' || 'CS_MASTER_SUM';
        EXCEPTION
           WHEN OTHERS THEN
              IF SQLCODE != -942 THEN
                 RAISE;
              END IF;
        END;    
    
2020-10-28 23:05:44,808 INFO sqlalchemy.engine.base.Engine {}
2020-10-28 23:05:44,923 INFO sqlalchemy.engine.base.Engine 
CREATE TABLE CS_MASTER_SUM AS
SELECT A.*,
((A."Max Dispense Date" - A."Min Dispense Date")+1) AS "Total Days",
(A."Count Script"/ A."Member Count") AS PR2,
(A."Sum Dispensed Quantity"/ A."Member Count") AS PR1,
(A."Count Script"/((A."Max Dispense Date" - A."Min Dispense Date")+1)) AS PR4,
(A."Sum Dispensed Quantity"/((A."Max Dispense Date" - A."Min Dispense Date")+1)) AS PR3
FROM
(SELECT
COUNT(DISTINCT CLM.MCAID_ID) AS "Member Count", --- unique beneficiaries
COUNT(CLM.RX_NBR) AS "Count Script", --- how many RXs written
SUM(CLM.DSPN_UNT_QTY) AS "Sum Dispensed Quantity",
MAX(CLM.DRUG

,Member Count,Count Script,Sum Dispensed Quantity,Max Dispense Date,Min Dispense Date,prscrb_prov_loc_id,Total Days,pr2,pr1,pr4,pr3
0,3,3,35,2020-06-28,2020-04-11,114093,79,1.000000,11.666667,0.037975,0.443038
1,13,39,1834,2020-06-30,2020-04-01,146072,91,3.000000,141.076923,0.428571,20.153846
2,9,15,933,2020-06-29,2020-04-02,165116,89,1.666667,103.666667,0.168539,10.483146
3,99,322,23754,2020-06-30,2020-04-01,1303,91,3.252525,239.939394,3.538462,261.032967
4,13,29,1555,2020-06-24,2020-04-03,107827,83,2.230769,119.615385,0.349398,18.734940


# Metric PR5

In [11]:
drop_table_if_exists('PR5')

p5_qry ="""
CREATE TABLE PR5 AS
SELECT A.PRSCRB_PROV_LOC_ID, A.CNT_CLM_2, B.CNT_CLM, 
(A.CNT_CLM_2/B.CNT_CLM) AS PR5

--CASE WHEN B.CNT_CLM < 5 THEN (A.CNT_CLM_2/B.CNT_CLM) * (B.CNT_CLM / 10)
--                ELSE  (A.CNT_CLM_2/B.CNT_CLM)
--                END AS PR5
FROM
(SELECT A.PRSCRB_PROV_LOC_ID, COUNT(DISTINCT ICN_NBR) AS CNT_CLM_2
FROM BIDM_USR_RPTS.CLM_LNE_FACT_V A
WHERE A.FED_DEA_DRUG_SCHDL_CD = '2' -- THIS CAN BE CHANGED TO LOOK AT ANY/ALL FED SCHEDS
AND A.CLM_PD_DT >= TO_DATE(\'{date1}\', 'MMDDYYYY')
AND A.CLM_FRST_SVC_DT BETWEEN TO_DATE(\'{date1}\', 'MMDDYYYY') AND TO_DATE(\'{date2}\', 'MMDDYYYY')
AND A.CURR_REC_IND = 'Y' 
AND A.SRC_REC_DEL_IND = 'N' 
AND A.MOST_RCNT_CLM_IND = 'Y'
AND A.ENC_IND = 'N' 
AND A.RVRSL_IND = 'N'
AND A.CLM_STS_CD = 'P' 
AND A.LNE_STS_CD = 'P'
AND A.CLM_TYP_CD IN ('P', 'Q')
GROUP BY A.PRSCRB_PROV_LOC_ID)A
INNER JOIN
(SELECT A.PRSCRB_PROV_LOC_ID, COUNT(DISTINCT ICN_NBR) AS CNT_CLM
FROM BIDM_USR_RPTS.CLM_LNE_FACT_V A
WHERE A.CLM_PD_DT >= TO_DATE(\'{date1}\', 'MMDDYYYY')
AND A.CLM_FRST_SVC_DT BETWEEN TO_DATE(\'{date1}\', 'MMDDYYYY') AND TO_DATE(\'{date2}\', 'MMDDYYYY')
AND A.CURR_REC_IND = 'Y' AND A.SRC_REC_DEL_IND = 'N' AND A.MOST_RCNT_CLM_IND = 'Y'
AND A.ENC_IND = 'N' AND A.RVRSL_IND = 'N'
AND A.CLM_STS_CD = 'P' AND A.LNE_STS_CD = 'P'
AND A.CLM_TYP_CD IN ('P', 'Q')
GROUP BY A.PRSCRB_PROV_LOC_ID)B
ON A.PRSCRB_PROV_LOC_ID = B.PRSCRB_PROV_LOC_ID
""".format(date1 = begin_date, date2 = end_date)

engine.execute(p5_qry)

p5_df = pd.read_sql('select * from PR5 where rownum < 10', engine)

p5_df.head()

2020-10-28 23:05:46,969 INFO sqlalchemy.engine.base.Engine 
        BEGIN
           EXECUTE IMMEDIATE 'DROP TABLE ' || 'PR5';
        EXCEPTION
           WHEN OTHERS THEN
              IF SQLCODE != -942 THEN
                 RAISE;
              END IF;
        END;    
    
2020-10-28 23:05:46,971 INFO sqlalchemy.engine.base.Engine {}
2020-10-28 23:05:47,085 INFO sqlalchemy.engine.base.Engine 
CREATE TABLE PR5 AS
SELECT A.PRSCRB_PROV_LOC_ID, A.CNT_CLM_2, B.CNT_CLM, 
(A.CNT_CLM_2/B.CNT_CLM) AS PR5

--CASE WHEN B.CNT_CLM < 5 THEN (A.CNT_CLM_2/B.CNT_CLM) * (B.CNT_CLM / 10)
--                ELSE  (A.CNT_CLM_2/B.CNT_CLM)
--                END AS PR5
FROM
(SELECT A.PRSCRB_PROV_LOC_ID, COUNT(DISTINCT ICN_NBR) AS CNT_CLM_2
FROM BIDM_USR_RPTS.CLM_LNE_FACT_V A
WHERE A.FED_DEA_DRUG_SCHDL_CD = '2' -- THIS CAN BE CHANGED TO LOOK AT ANY/ALL FED SCHEDS
AND A.CLM_PD_DT >= TO_DATE('04012020', 'MMDDYYYY')
AND A.CLM_FRST_SVC_DT BETWEEN TO_DATE('04012020', 'MMDDYYYY') AND TO_DATE('06302020', 'MMDDYYY

,prscrb_prov_loc_id,cnt_clm_2,cnt_clm,pr5
0,109761,70,350,0.200000
1,2755,434,1157,0.375108
2,119938,45,413,0.108959
3,128032,116,1712,0.067757
4,106006,91,1284,0.070872


# Metric PR6

In [12]:
drop_table_if_exists('PR6')


p6_qry ="""
CREATE TABLE PR6 AS

WITH MEM_VALID AS
(
SELECT DISTINCT C.MCAID_ID AS MEMBER_VALID
FROM BIDM_USR_RPTS.CLM_LNE_FACT_V C
WHERE ((C.DIAG_1_CD IN (SELECT "CNCR_DX_CD" FROM D98JMOHR.DX_CANCER) OR --- CANCER DIAG
     C.DIAG_2_CD IN (SELECT "CNCR_DX_CD" FROM D98JMOHR.DX_CANCER) OR
     C.DIAG_3_CD IN (SELECT "CNCR_DX_CD" FROM D98JMOHR.DX_CANCER) OR
     C.DIAG_4_CD IN (SELECT "CNCR_DX_CD" FROM D98JMOHR.DX_CANCER))
OR TO_CHAR(C.RVN_CD) IN (SELECT TO_CHAR("Revenue Code") FROM D98JMOHR.DX_HOSPICE_REV) -- HOSPICE PROC
OR TO_CHAR(C.PROC_CD) IN (SELECT TO_CHAR("Procedure Code") FROM D98JMOHR.DX_HOSPICE_PROC)) -- HOSPICE PROC
--AND C.REND_PROV_LOC_ID = '115182'
AND C.CLM_PD_DT >= TO_DATE(\'{date1}\','MMDDYYYY')
AND C.CLM_FRST_SVC_DT BETWEEN TO_DATE(\'{date1}\', 'MMDDYYYY') AND TO_DATE(\'{date2}\', 'MMDDYYYY')
AND C.CURR_REC_IND = 'Y'
AND C.SRC_REC_DEL_IND = 'N'
AND C.MOST_RCNT_CLM_IND = 'Y'
AND C.CLM_STS_CD = 'P'
AND C.LNE_STS_CD = 'P'
)

SELECT VAL.*, COUNT(DISTINCT MAS.MCAID_ID) AS MEM_TOT, 
(VAL.CNT_VALID/COUNT(DISTINCT MAS.MCAID_ID)) AS PR6
--(VAL.CNT_VALID) AS PR6
FROM 
(SELECT M.PRSCRB_PROV_LOC_ID, COUNT(DISTINCT M.MCAID_ID) AS CNT_VALID
FROM CS_MASTER M
WHERE M.MCAID_ID IN (SELECT DISTINCT MEMBER_VALID FROM MEM_VALID )
GROUP BY M.PRSCRB_PROV_LOC_ID) VAL
INNER JOIN CS_MASTER MAS
ON VAL.PRSCRB_PROV_LOC_ID = MAS.PRSCRB_PROV_LOC_ID
GROUP BY VAL.PRSCRB_PROV_LOC_ID, VAL. CNT_VALID
""".format(date1 = begin_date, date2 = end_date)

engine.execute(p6_qry)


p6_df = pd.read_sql('select * from PR6 where rownum < 10', engine)

p6_df.head()

2020-10-28 23:06:14,741 INFO sqlalchemy.engine.base.Engine 
        BEGIN
           EXECUTE IMMEDIATE 'DROP TABLE ' || 'PR6';
        EXCEPTION
           WHEN OTHERS THEN
              IF SQLCODE != -942 THEN
                 RAISE;
              END IF;
        END;    
    
2020-10-28 23:06:14,743 INFO sqlalchemy.engine.base.Engine {}
2020-10-28 23:06:14,949 INFO sqlalchemy.engine.base.Engine 
CREATE TABLE PR6 AS

WITH MEM_VALID AS
(
SELECT DISTINCT C.MCAID_ID AS MEMBER_VALID
FROM BIDM_USR_RPTS.CLM_LNE_FACT_V C
WHERE ((C.DIAG_1_CD IN (SELECT "CNCR_DX_CD" FROM D98JMOHR.DX_CANCER) OR --- CANCER DIAG
     C.DIAG_2_CD IN (SELECT "CNCR_DX_CD" FROM D98JMOHR.DX_CANCER) OR
     C.DIAG_3_CD IN (SELECT "CNCR_DX_CD" FROM D98JMOHR.DX_CANCER) OR
     C.DIAG_4_CD IN (SELECT "CNCR_DX_CD" FROM D98JMOHR.DX_CANCER))
OR TO_CHAR(C.RVN_CD) IN (SELECT TO_CHAR("Revenue Code") FROM D98JMOHR.DX_HOSPICE_REV) -- HOSPICE PROC
OR TO_CHAR(C.PROC_CD) IN (SELECT TO_CHAR("Procedure Code") FROM D98JMOHR.DX_HOSPICE_

,prscrb_prov_loc_id,cnt_valid,mem_tot,pr6
0,103869,2,43,0.046512
1,121692,1,96,0.010417
2,131073,2,20,0.100000
3,170301,1,18,0.055556
4,156749,1,6,0.166667


# Metric PR7

## increasing mme over time

In [13]:
p7_qry = """
select prscrb_prov_loc_id, member_id,
LISTAGG(round(mme_per_day,2), ',') WITHIN GROUP (ORDER BY dispence_date) AS mme_per_day
from
(select 
prscrb_prov_loc_id
,mcaid_id as member_id
,drug_dspn_dt as dispence_date
,"Strength_Per_Unit" * (csm."DSPN_UNT_QTY" / csm."DRUG_DAY_SPLY_QTY") * "MME_Conversion_Factor" AS mme_per_day
from CS_MASTER csm
where 1=1
and "MME_Conversion_Factor" is not null
order by member_id, dispence_date) aa
group by prscrb_prov_loc_id, member_id
"""

# Setup mme per day array
p7_df_temp1 = pd.read_sql(p7_qry, engine)
p7_df_temp1['mme_per_day'] = p7_df_temp1['mme_per_day'].apply(lambda x: x.split(',')).apply(lambda y: [float(i) for i in y])
p7_df_temp1['num_obs'] = p7_df_temp1['mme_per_day'].apply(len)

# regression formula to see if increasing
def regress(y):
    if(len(y)==1):
        return(0)
    else:
        x = list(range(1,len(y)+1))
        out = stats.linregress(x,y)
        return(out.slope)

# Find which members are increasing in MME    
p7_df_temp1['slope'] = p7_df_temp1['mme_per_day'].apply(regress)
p7_df_temp1['increasing'] = p7_df_temp1['slope']>0

# For each provider, count the number of members who's MME per day increased
p7_df_temp2 = p7_df_temp1[p7_df_temp1.increasing==True]. \
    reset_index(). \
    groupby(['prscrb_prov_loc_id'], as_index = False). \
    agg({'increasing': 'count'}). \
    rename(columns={"increasing": "pr7"})

# p7_df_temp2.prscrb_prov_loc_id = p7_df_temp2.prscrb_prov_loc_id.to_string()

# Write back to DB
drop_table_if_exists('PR7')
p7_df_temp2.to_sql('PR7', engine, if_exists='replace', index=False, dtype={"prscrb_prov_loc_id": types.VARCHAR(256)})

p7_df = pd.read_sql('select * from PR7 where rownum < 10', engine)

p7_df.head()

2020-10-28 23:08:43,882 INFO sqlalchemy.engine.base.Engine SELECT table_name FROM all_tables WHERE table_name = :name AND owner = :schema_name
2020-10-28 23:08:43,884 INFO sqlalchemy.engine.base.Engine {'name': '\nselect prscrb_prov_loc_id, member_id,\nLISTAGG(round(mme_per_day,2), \',\') WITHIN GROUP (ORDER BY dispence_date) AS mme_per_day\nfrom\n(select \np ... (189 characters truncated) ... om CS_MASTER csm\nwhere 1=1\nand "MME_Conversion_Factor" is not null\norder by member_id, dispence_date) aa\ngroup by prscrb_prov_loc_id, member_id\n', 'schema_name': 'U4J8175'}
2020-10-28 23:08:44,855 INFO sqlalchemy.engine.base.Engine 
select prscrb_prov_loc_id, member_id,
LISTAGG(round(mme_per_day,2), ',') WITHIN GROUP (ORDER BY dispence_date) AS mme_per_day
from
(select 
prscrb_prov_loc_id
,mcaid_id as member_id
,drug_dspn_dt as dispence_date
,"Strength_Per_Unit" * (csm."DSPN_UNT_QTY" / csm."DRUG_DAY_SPLY_QTY") * "MME_Conversion_Factor" AS mme_per_day
from CS_MASTER csm
where 1=1
and "MME_Con

/opt/conda/envs/Python-3.6-WMLCE/lib/python3.6/site-packages/pandas/io/sql.py:1191: UserWarning: The provided table name 'PR7' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  warnings.warn(msg, UserWarning)


,prscrb_prov_loc_id,pr7
0,135770,1
1,135784,1
2,135820,1
3,135824,1
4,135850,4


# Metric PR8


In [14]:
drop_table_if_exists('PR8')

p8_qry ="""
CREATE TABLE PR8 AS
SELECT OP.PRSCRB_PROV_LOC_ID, OP.OPOID_DEATHS, COUNT(DISTINCT CLM.MCAID_ID) AS CNT_MEM,
(OP.OPOID_DEATHS/COUNT(DISTINCT CLM.MCAID_ID)) AS PR8
--OP.OPOID_DEATHS as PR8

FROM BIDM_USR_RPTS.CLM_LNE_FACT_V CLM
INNER JOIN
(SELECT A.PRSCRB_PROV_LOC_ID, COUNT(DISTINCT A.MCAID_ID) AS OPOID_DEATHS
FROM BIDM_USR_RPTS.CLM_LNE_FACT_V A
INNER JOIN D98JMOHR.OPIOID_DEATHS B -- NEEDED BECAUSE THE SOURCE DATA IS NOT IN UAT.  CAN BE ADJUSTED WHEN IMPLEMENTED IN PROD
ON A.MCAID_ID = B.CLNT_ID
INNER JOIN D98JMOHR.CDC_WKBK_2018 W 
ON A.NDC_CD = W.NDC
AND W."Class" = 'Opioid'
WHERE
 A.CLM_TYP_CD IN ('P', 'Q')
AND A.MOST_RCNT_CLM_IND = 'Y' 
AND A.CURR_REC_IND = 'Y' 
AND A.SRC_REC_DEL_IND = 'N'
AND B.DOD BETWEEN A.DRUG_DSPN_DT AND A.DRUG_DSPN_DT + 30
GROUP BY A.PRSCRB_PROV_LOC_ID) OP
ON CLM.PRSCRB_PROV_LOC_ID = OP.PRSCRB_PROV_LOC_ID
GROUP BY OP.PRSCRB_PROV_LOC_ID, OP.OPOID_DEATHS
""".format(date1 = begin_date, date2 = end_date)

engine.execute(p8_qry)

p8_df = pd.read_sql('select * from PR8 where rownum < 10', engine)

p8_df.head()

2020-10-28 23:09:33,534 INFO sqlalchemy.engine.base.Engine 
        BEGIN
           EXECUTE IMMEDIATE 'DROP TABLE ' || 'PR8';
        EXCEPTION
           WHEN OTHERS THEN
              IF SQLCODE != -942 THEN
                 RAISE;
              END IF;
        END;    
    
2020-10-28 23:09:33,536 INFO sqlalchemy.engine.base.Engine {}
2020-10-28 23:09:33,645 INFO sqlalchemy.engine.base.Engine 
CREATE TABLE PR8 AS
SELECT OP.PRSCRB_PROV_LOC_ID, OP.OPOID_DEATHS, COUNT(DISTINCT CLM.MCAID_ID) AS CNT_MEM,
(OP.OPOID_DEATHS/COUNT(DISTINCT CLM.MCAID_ID)) AS PR8
--OP.OPOID_DEATHS as PR8

FROM BIDM_USR_RPTS.CLM_LNE_FACT_V CLM
INNER JOIN
(SELECT A.PRSCRB_PROV_LOC_ID, COUNT(DISTINCT A.MCAID_ID) AS OPOID_DEATHS
FROM BIDM_USR_RPTS.CLM_LNE_FACT_V A
INNER JOIN D98JMOHR.OPIOID_DEATHS B -- NEEDED BECAUSE THE SOURCE DATA IS NOT IN UAT.  CAN BE ADJUSTED WHEN IMPLEMENTED IN PROD
ON A.MCAID_ID = B.CLNT_ID
INNER JOIN D98JMOHR.CDC_WKBK_2018 W 
ON A.NDC_CD = W.NDC
AND W."Class" = 'Opioid'
WHERE
 A.CLM_TYP_C

,prscrb_prov_loc_id,opoid_deaths,cnt_mem,pr8
0,3469,1,239,0.004184
1,124557,1,646,0.001548
2,142446,1,1664,0.000601
3,132740,2,3472,0.000576
4,100220,2,989,0.002022


# Metric PR9

In [15]:
drop_table_if_exists('PR9')

p9_qry ="""
CREATE TABLE PR9 AS
WITH FULL_MORB AS
(
SELECT DISTINCT C.MCAID_ID, C.DRUG_DSPN_DT, C.PRSCRB_PROV_LOC_ID
FROM BIDM_USR_RPTS.CLM_LNE_FACT_V C
INNER JOIN D98JMOHR.CDC_WKBK_2018 W ON C.NDC_CD = W.NDC
WHERE C.CLM_PD_DT >= TO_DATE(\'{date1}\', 'MMDDYYYY')
AND C.DRUG_DSPN_DT BETWEEN TO_DATE(\'{date1}\', 'MMDDYYYY') AND TO_DATE(\'{date2}\','MMDDYYYY')
AND C.CLM_TYP_CD IN ('P', 'Q')
AND C.MOST_RCNT_CLM_IND = 'Y' AND C.CURR_REC_IND = 'Y' AND C.SRC_REC_DEL_IND = 'N'
AND W."Class" = 'Opioid') --- CAN BE CHANGED TO ANY CONTROLLED SUBSTANCES
,

FULL_MORB_RESULTS AS
((SELECT DISTINCT C.MCAID_ID, C.CLM_FRST_SVC_DT, L.PRSCRB_PROV_LOC_ID, -- BECAUSE OF DATA STRUCTURE, HAVE TO DO THIS Q FOR EACH DIAG CODE (THERE ARE 4)
                CASE WHEN SUBSTR(C.DIAG_1_CD, 4, 1) = '0' THEN 'Opium'
                           WHEN SUBSTR(C.DIAG_1_CD, 4, 1) = '1' THEN 'Heroin'
                           WHEN SUBSTR(C.DIAG_1_CD, 4, 1) = '2' THEN 'Natural and semisynthetic opioids'
                           WHEN SUBSTR(C.DIAG_1_CD, 4, 1) = '3' THEN 'Methadone'
                           WHEN SUBSTR(C.DIAG_1_CD, 4, 1) = '4' THEN 'Synthetic opioids, other than methadone'
                           WHEN SUBSTR(C.DIAG_1_CD, 4, 1) = '6' THEN 'Other and unspecified narcotics'
                           END AS RELATED_DRUG,
                      CASE WHEN SUBSTR(C.DIAG_1_CD, 6, 1) = '1' THEN 'Accidental'
                           WHEN SUBSTR(C.DIAG_1_CD, 6, 1) = '2' THEN 'Intentional Self-Harm'
                           WHEN SUBSTR(C.DIAG_1_CD, 6, 1) = '3' THEN 'Assault'
                           WHEN SUBSTR(C.DIAG_1_CD, 6, 1) = '4' THEN 'Undetermined'
                           ELSE 'Undetermined'
                           END AS "Cause",
                      CASE WHEN SUBSTR(C.DIAG_1_CD, 7, 1) = 'D' THEN 'Subsequent'
                           WHEN SUBSTR(C.DIAG_1_CD, 7, 1) = 'A' THEN 'Initial'
                           ELSE 'N/A'
                           END AS "Subsequent or Initial"
FROM BIDM_USR_RPTS.CLM_LNE_FACT_V C
INNER JOIN FULL_MORB L ON L.MCAID_ID = C.MCAID_ID
WHERE C.CLM_PD_DT >= TO_DATE(\'{date1}\', 'MMDDYYYY')
AND C.CLM_FRST_SVC_DT >= TO_DATE(\'{date1}\', 'MMDDYYYY')
AND C.PROC_CD IN ('99283', '99284', '99285') --Moderate to Severe Emergency Room Visits
AND C.CURR_REC_IND = 'Y' AND C.SRC_REC_DEL_IND = 'N' AND C.MOST_RCNT_CLM_IND = 'Y'
AND C.DIAG_1_CD LIKE 'T40%'
AND SUBSTR(C.DIAG_1_CD, 4, 1) IN ('0', '1', '2', '3', '4', '6') --Opioid overdose related
AND C.DIAG_1_SEQ_CD IN ('01', 'A', 'E') --Primary, Admitting, Emergency
AND SUBSTR(C.DIAG_1_CD, 6, 1) IN ('1', '2', '3', '4')
AND SUBSTR(C.DIAG_1_CD, 7, 1) <> 'S' --Exclude indirect complications
AND C.CLM_FRST_SVC_DT BETWEEN L.DRUG_DSPN_DT AND L.DRUG_DSPN_DT + 30)

UNION

(SELECT DISTINCT C.MCAID_ID, C.CLM_FRST_SVC_DT, L.PRSCRB_PROV_LOC_ID,
                CASE WHEN SUBSTR(C.DIAG_2_CD, 4, 1) = '0' THEN 'Opium'
                           WHEN SUBSTR(C.DIAG_2_CD, 4, 1) = '1' THEN 'Heroin'
                           WHEN SUBSTR(C.DIAG_2_CD, 4, 1) = '2' THEN 'Natural and semisynthetic opioids'
                           WHEN SUBSTR(C.DIAG_2_CD, 4, 1) = '3' THEN 'Methadone'
                           WHEN SUBSTR(C.DIAG_2_CD, 4, 1) = '4' THEN 'Synthetic opioids, other than methadone'
                           WHEN SUBSTR(C.DIAG_2_CD, 4, 1) = '6' THEN 'Other and unspecified narcotics'
                           END AS RELATED_DRUG,
                      CASE WHEN SUBSTR(C.DIAG_2_CD, 6, 1) = '1' THEN 'Accidental'
                           WHEN SUBSTR(C.DIAG_2_CD, 6, 1) = '2' THEN 'Intentional Self-Harm'
                           WHEN SUBSTR(C.DIAG_2_CD, 6, 1) = '3' THEN 'Assault'
                           WHEN SUBSTR(C.DIAG_2_CD, 6, 1) = '4' THEN 'Undetermined'
                           ELSE 'Undetermined'
                           END AS "Cause",
                      CASE WHEN SUBSTR(C.DIAG_2_CD, 7, 1) = 'D' THEN 'Subsequent'
                           WHEN SUBSTR(C.DIAG_2_CD, 7, 1) = 'A' THEN 'Initial'
                           ELSE 'N/A'
                           END AS "Subsequent or Initial"
FROM BIDM_USR_RPTS.CLM_LNE_FACT_V C
INNER JOIN FULL_MORB L ON L.MCAID_ID = C.MCAID_ID
WHERE C.CLM_PD_DT >= TO_DATE(\'{date1}\', 'MMDDYYYY')
AND C.CLM_FRST_SVC_DT >= TO_DATE(\'{date1}\', 'MMDDYYYY')
AND C.PROC_CD IN ('99283', '99284', '99285') --Moderate to Severe Emergency Room Visits
AND C.CURR_REC_IND = 'Y' AND C.SRC_REC_DEL_IND = 'N' AND C.MOST_RCNT_CLM_IND = 'Y'
AND C.DIAG_2_CD LIKE 'T40%'
AND SUBSTR(C.DIAG_2_CD, 4, 1) IN ('0', '1', '2', '3', '4', '6') --Opioid overdose related
AND C.DIAG_2_SEQ_CD IN ('01', 'A', 'E') --Primary, Admitting, Emergency
AND SUBSTR(C.DIAG_2_CD, 6, 1) IN ('1', '2', '3', '4')
AND SUBSTR(C.DIAG_2_CD, 7, 1) <> 'S' --Exclude indirect complications
AND C.CLM_FRST_SVC_DT BETWEEN L.DRUG_DSPN_DT AND L.DRUG_DSPN_DT + 30)

UNION

(SELECT DISTINCT C.MCAID_ID, C.CLM_FRST_SVC_DT, L.PRSCRB_PROV_LOC_ID,
                CASE WHEN SUBSTR(C.DIAG_3_CD, 4, 1) = '0' THEN 'Opium'
                           WHEN SUBSTR(C.DIAG_3_CD, 4, 1) = '1' THEN 'Heroin'
                           WHEN SUBSTR(C.DIAG_3_CD, 4, 1) = '2' THEN 'Natural and semisynthetic opioids'
                           WHEN SUBSTR(C.DIAG_3_CD, 4, 1) = '3' THEN 'Methadone'
                           WHEN SUBSTR(C.DIAG_3_CD, 4, 1) = '4' THEN 'Synthetic opioids, other than methadone'
                           WHEN SUBSTR(C.DIAG_3_CD, 4, 1) = '6' THEN 'Other and unspecified narcotics'
                           END AS RELATED_DRUG,
                      CASE WHEN SUBSTR(C.DIAG_3_CD, 6, 1) = '1' THEN 'Accidental'
                           WHEN SUBSTR(C.DIAG_3_CD, 6, 1) = '2' THEN 'Intentional Self-Harm'
                           WHEN SUBSTR(C.DIAG_3_CD, 6, 1) = '3' THEN 'Assault'
                           WHEN SUBSTR(C.DIAG_3_CD, 6, 1) = '4' THEN 'Undetermined'
                           ELSE 'Undetermined'
                           END AS "Cause",
                      CASE WHEN SUBSTR(C.DIAG_3_CD, 7, 1) = 'D' THEN 'Subsequent'
                           WHEN SUBSTR(C.DIAG_3_CD, 7, 1) = 'A' THEN 'Initial'
                           ELSE 'N/A'
                           END AS "Subsequent or Initial"
FROM BIDM_USR_RPTS.CLM_LNE_FACT_V C
INNER JOIN FULL_MORB L ON L.MCAID_ID = C.MCAID_ID
WHERE C.CLM_PD_DT >= TO_DATE(\'{date1}\', 'MMDDYYYY')
AND C.CLM_FRST_SVC_DT >= TO_DATE(\'{date1}\', 'MMDDYYYY')
AND C.PROC_CD IN ('99283', '99284', '99285') --Moderate to Severe Emergency Room Visits
AND C.CURR_REC_IND = 'Y' AND C.SRC_REC_DEL_IND = 'N' AND C.MOST_RCNT_CLM_IND = 'Y'
AND C.DIAG_3_CD LIKE 'T40%'
AND SUBSTR(C.DIAG_3_CD, 4, 1) IN ('0', '1', '2', '3', '4', '6') --Opioid overdose related
AND C.DIAG_3_SEQ_CD IN ('01', 'A', 'E') --Primary, Admitting, Emergency
AND SUBSTR(C.DIAG_3_CD, 6, 1) IN ('1', '2', '3', '4')
AND SUBSTR(C.DIAG_3_CD, 7, 1) <> 'S' --Exclude indirect complications
AND C.CLM_FRST_SVC_DT BETWEEN L.DRUG_DSPN_DT AND L.DRUG_DSPN_DT + 30)

UNION

(SELECT DISTINCT C.MCAID_ID, C.CLM_FRST_SVC_DT, L.PRSCRB_PROV_LOC_ID,
                CASE WHEN SUBSTR(C.DIAG_4_CD, 4, 1) = '0' THEN 'Opium'
                           WHEN SUBSTR(C.DIAG_4_CD, 4, 1) = '1' THEN 'Heroin'
                           WHEN SUBSTR(C.DIAG_4_CD, 4, 1) = '2' THEN 'Natural and semisynthetic opioids'
                           WHEN SUBSTR(C.DIAG_4_CD, 4, 1) = '3' THEN 'Methadone'
                           WHEN SUBSTR(C.DIAG_4_CD, 4, 1) = '4' THEN 'Synthetic opioids, other than methadone'
                           WHEN SUBSTR(C.DIAG_4_CD, 4, 1) = '6' THEN 'Other and unspecified narcotics'
                           END AS RELATED_DRUG,
                      CASE WHEN SUBSTR(C.DIAG_4_CD, 6, 1) = '1' THEN 'Accidental'
                           WHEN SUBSTR(C.DIAG_4_CD, 6, 1) = '2' THEN 'Intentional Self-Harm'
                           WHEN SUBSTR(C.DIAG_4_CD, 6, 1) = '3' THEN 'Assault'
                           WHEN SUBSTR(C.DIAG_4_CD, 6, 1) = '4' THEN 'Undetermined'
                           ELSE 'Undetermined'
                           END AS "Cause",
                      CASE WHEN SUBSTR(C.DIAG_4_CD, 7, 1) = 'D' THEN 'Subsequent'
                           WHEN SUBSTR(C.DIAG_4_CD, 7, 1) = 'A' THEN 'Initial'
                           ELSE 'N/A'
                           END AS "Subsequent or Initial"
FROM BIDM_USR_RPTS.CLM_LNE_FACT_V C
INNER JOIN FULL_MORB L ON L.MCAID_ID = C.MCAID_ID
WHERE C.CLM_PD_DT >= TO_DATE(\'{date1}\', 'MMDDYYYY')
AND C.CLM_FRST_SVC_DT >= TO_DATE(\'{date1}\', 'MMDDYYYY')
AND C.PROC_CD IN ('99283', '99284', '99285') --Moderate to Severe Emergency Room Visits
AND C.CURR_REC_IND = 'Y' AND C.SRC_REC_DEL_IND = 'N' AND C.MOST_RCNT_CLM_IND = 'Y'
AND C.DIAG_4_CD LIKE 'T40%'
AND SUBSTR(C.DIAG_4_CD, 4, 1) IN ('0', '1', '2', '3', '4', '6') --Opioid overdose related
AND C.DIAG_4_SEQ_CD IN ('01', 'A', 'E') --Primary, Admitting, Emergency
AND SUBSTR(C.DIAG_4_CD, 6, 1) IN ('1', '2', '3', '4')
AND SUBSTR(C.DIAG_4_CD, 7, 1) <> 'S' --Exclude indirect complications
AND C.CLM_FRST_SVC_DT BETWEEN L.DRUG_DSPN_DT AND L.DRUG_DSPN_DT + 30))

SELECT A.PRSCRB_PROV_LOC_ID, COUNT(DISTINCT A.MCAID_ID) AS COUNT_MORB, 
COUNT(DISTINCT B.MCAID_ID) AS CNT_MEM,
COUNT(DISTINCT A.MCAID_ID)/COUNT(DISTINCT B.MCAID_ID) AS PR9
--COUNT(DISTINCT A.MCAID_ID) as PR9
FROM FULL_MORB_RESULTS A
INNER JOIN FULL_MORB B
ON A.PRSCRB_PROV_LOC_ID = B.PRSCRB_PROV_LOC_ID
GROUP BY A.PRSCRB_PROV_LOC_ID
""".format(date1 = begin_date, date2 = end_date)

engine.execute(p9_qry)

p9_df = pd.read_sql('select * from PR9 where rownum < 10', engine)

p9_df.head()

2020-10-28 23:11:01,627 INFO sqlalchemy.engine.base.Engine 
        BEGIN
           EXECUTE IMMEDIATE 'DROP TABLE ' || 'PR9';
        EXCEPTION
           WHEN OTHERS THEN
              IF SQLCODE != -942 THEN
                 RAISE;
              END IF;
        END;    
    
2020-10-28 23:11:01,629 INFO sqlalchemy.engine.base.Engine {}
2020-10-28 23:11:01,859 INFO sqlalchemy.engine.base.Engine 
CREATE TABLE PR9 AS
WITH FULL_MORB AS
(
SELECT DISTINCT C.MCAID_ID, C.DRUG_DSPN_DT, C.PRSCRB_PROV_LOC_ID
FROM BIDM_USR_RPTS.CLM_LNE_FACT_V C
INNER JOIN D98JMOHR.CDC_WKBK_2018 W ON C.NDC_CD = W.NDC
WHERE C.CLM_PD_DT >= TO_DATE('04012020', 'MMDDYYYY')
AND C.DRUG_DSPN_DT BETWEEN TO_DATE('04012020', 'MMDDYYYY') AND TO_DATE('06302020','MMDDYYYY')
AND C.CLM_TYP_CD IN ('P', 'Q')
AND C.MOST_RCNT_CLM_IND = 'Y' AND C.CURR_REC_IND = 'Y' AND C.SRC_REC_DEL_IND = 'N'
AND W."Class" = 'Opioid') --- CAN BE CHANGED TO ANY CONTROLLED SUBSTANCES
,

FULL_MORB_RESULTS AS
((SELECT DISTINCT C.MCAID_ID, C.CLM_FRST_SV

2020-10-28 23:11:01,861 INFO sqlalchemy.engine.base.Engine {}
2020-10-28 23:12:39,466 INFO sqlalchemy.engine.base.Engine COMMIT
2020-10-28 23:12:40,323 INFO sqlalchemy.engine.base.Engine SELECT table_name FROM all_tables WHERE table_name = :name AND owner = :schema_name
2020-10-28 23:12:40,324 INFO sqlalchemy.engine.base.Engine {'name': 'select * from PR9 where rownum < 10', 'schema_name': 'U4J8175'}
2020-10-28 23:12:40,409 INFO sqlalchemy.engine.base.Engine select * from PR9 where rownum < 10
2020-10-28 23:12:40,410 INFO sqlalchemy.engine.base.Engine {}


,prscrb_prov_loc_id,count_morb,cnt_mem,pr9
0,141226,1,1,1.000000
1,147110,1,62,0.016129
2,154660,1,26,0.038462
3,177706,1,63,0.015873
4,178627,1,100,0.010000


# Metric PR10

In [16]:
drop_table_if_exists('PR10')

p10_qry ="""
CREATE TABLE PR10 AS
WITH TRIN_TEMP AS
(SELECT OP.*
, D.*
FROM
(SELECT  A."Class" AS "OP Drug Class", --- REGION 1: CONCURRENT OPOID AND OTHERS
  A.DRUG_DSPN_DT AS OP_DRUG_DSPN_DT,
  A.PRSRB_DT AS OP_PRSRB_DT,
  A.MCAID_ID AS OP_MCAID_ID,
  A.ICN_NBR AS OP_ICN_NBR,
  A.LNE_NBR AS OP_LNE_NBR,
  A.PRSCRB_PROV_LOC_ID AS OP_PRSCRB_PROV_LOC_ID
FROM CS_MASTER A
WHERE A."Class" = 'Opioid')OP
INNER JOIN(
SELECT  A."Class" AS "Drug Class",
  A.DRUG_DSPN_DT,
  A.PRSRB_DT,
  A.MCAID_ID,
  A.ICN_NBR,
  A.LNE_NBR,
  A.PRSCRB_PROV_LOC_ID
FROM CS_MASTER A
WHERE A."Class" IN ('Benzo', 'Muscle Relaxant', 'Misc')
) D
ON OP.OP_MCAID_ID = D.MCAID_ID
AND D.PRSRB_DT BETWEEN (OP.OP_PRSRB_DT - 7) AND (OP.OP_PRSRB_DT +7))
,
TRIN_TEMP_CNT AS
(SELECT M.OP_PRSCRB_PROV_LOC_ID, M.OP_ICN_NBR, M.OP_MCAID_ID, COUNT(M.OP_ICN_NBR) AS CNT_ICN --- REGION 2: COUNTING OPOID CLAIMS.  CLAIMS WITH MORE THAN ONE INSTACNE, HAVE TRINITYS
FROM TRIN_TEMP M--- END REGION 1
GROUP BY M.OP_PRSCRB_PROV_LOC_ID, M.OP_ICN_NBR, M.OP_MCAID_ID
HAVING COUNT(M.OP_ICN_NBR) > 1
ORDER BY M.OP_PRSCRB_PROV_LOC_ID, M.OP_MCAID_ID, M.OP_ICN_NBR)
SELECT A.OP_PRSCRB_PROV_LOC_ID, COUNT(DISTINCT A.OP_MCAID_ID) AS CNT_TRIN, COUNT(DISTINCT B.MCAID_ID) AS CNT_CLNT,
(COUNT(DISTINCT A.OP_MCAID_ID)/COUNT(DISTINCT B.MCAID_ID)) AS PR10
FROM
(SELECT A."OP Drug Class", A.OP_DRUG_DSPN_DT, A.OP_PRSRB_DT, A.OP_MCAID_ID, A.OP_ICN_NBR, A.OP_LNE_NBR, A.OP_PRSCRB_PROV_LOC_ID
FROM TRIN_TEMP A
INNER JOIN (TRIN_TEMP_CNT) B
ON A.OP_ICN_NBR = B.OP_ICN_NBR
UNION ALL
SELECT A."Drug Class",
  A.DRUG_DSPN_DT,
  A.PRSRB_DT,
  A.MCAID_ID,
  A.ICN_NBR,
  A.LNE_NBR,
  A.PRSCRB_PROV_LOC_ID
FROM TRIN_TEMP A
INNER JOIN (TRIN_TEMP_CNT) B
ON A.OP_ICN_NBR = B.OP_ICN_NBR) A
INNER JOIN CS_MASTER B
ON A.OP_PRSCRB_PROV_LOC_ID = B.PRSCRB_PROV_LOC_ID
GROUP BY A.OP_PRSCRB_PROV_LOC_ID
""".format(date1 = begin_date, date2 = end_date)

engine.execute(p10_qry)

p10_df = pd.read_sql('select * from PR10 where rownum < 10', engine)

p10_df.head()

2020-10-28 23:12:40,585 INFO sqlalchemy.engine.base.Engine 
        BEGIN
           EXECUTE IMMEDIATE 'DROP TABLE ' || 'PR10';
        EXCEPTION
           WHEN OTHERS THEN
              IF SQLCODE != -942 THEN
                 RAISE;
              END IF;
        END;    
    
2020-10-28 23:12:40,587 INFO sqlalchemy.engine.base.Engine {}
2020-10-28 23:12:40,821 INFO sqlalchemy.engine.base.Engine 
CREATE TABLE PR10 AS
WITH TRIN_TEMP AS
(SELECT OP.*
, D.*
FROM
(SELECT  A."Class" AS "OP Drug Class", --- REGION 1: CONCURRENT OPOID AND OTHERS
  A.DRUG_DSPN_DT AS OP_DRUG_DSPN_DT,
  A.PRSRB_DT AS OP_PRSRB_DT,
  A.MCAID_ID AS OP_MCAID_ID,
  A.ICN_NBR AS OP_ICN_NBR,
  A.LNE_NBR AS OP_LNE_NBR,
  A.PRSCRB_PROV_LOC_ID AS OP_PRSCRB_PROV_LOC_ID
FROM CS_MASTER A
WHERE A."Class" = 'Opioid')OP
INNER JOIN(
SELECT  A."Class" AS "Drug Class",
  A.DRUG_DSPN_DT,
  A.PRSRB_DT,
  A.MCAID_ID,
  A.ICN_NBR,
  A.LNE_NBR,
  A.PRSCRB_PROV_LOC_ID
FROM CS_MASTER A
WHERE A."Class" IN ('Benzo', 'Muscle Relaxant', 'Mi

,op_prscrb_prov_loc_id,cnt_trin,cnt_clnt,pr10
0,100016,2,12,0.166667
1,100023,1,4,0.250000
2,100027,1,5,0.200000
3,100052,1,7,0.142857
4,100097,1,9,0.111111


In [17]:
# ids = pd.read_sql('select mcaid_id from CS_MASTER where prscrb_prov_loc_id = 100052', engine)
# ids

In [18]:
# list(ids.mcaid_id)

In [19]:
# qqq = """
# select mcaid_id, "Class", drug_dspn_dt
# from CS_MASTER
# where 1=1
# and "Class" in ('Opioid', 'Benzo', 'Muscle Relaxant')
# and mcaid_id in ('P053490', 'Y491292', 'Q957903', 'Q957903', 'Q957903', 'O067759', 'P053490')
# order by mcaid_id, drug_dspn_dt
# """

# # Setup mme per day array
# temp = pd.read_sql(qqq, engine)

# temp

# Metric PR11

In [20]:
drop_table_if_exists('PR11')

pr11_qry ="""
CREATE TABLE PR11 AS
SELECT O.PRSCRB_PROV_LOC_ID, O.CNT_OFFICE, COUNT(DISTINCT M.CLM_LNE_FACT_SK) AS CNT_TOT,
(O.CNT_OFFICE/COUNT(DISTINCT M.CLM_LNE_FACT_SK)) AS PR11

--CASE WHEN COUNT(DISTINCT M.CLM_LNE_FACT_SK) < 10 THEN (O.CNT_OFFICE/COUNT(DISTINCT M.CLM_LNE_FACT_SK)) * (COUNT(DISTINCT M.CLM_LNE_FACT_SK) / 10)
--                ELSE  (O.CNT_OFFICE/COUNT(DISTINCT M.CLM_LNE_FACT_SK))
--                END AS PR11


FROM CS_MASTER M
INNER JOIN
(SELECT A.PRSCRB_PROV_LOC_ID, COUNT(DISTINCT A.CLM_LNE_FACT_SK) AS CNT_OFFICE
FROM BIDM_USR_RPTS.CLM_LNE_FACT_V B
INNER JOIN CS_MASTER A
ON B.LNE_FRST_SVC_DT = A.PRSRB_DT
AND B.MCAID_ID = A.MCAID_ID
WHERE B.PROC_CD BETWEEN '99201' AND '99499'
AND B.CLM_FRST_SVC_DT BETWEEN TO_DATE (\'{date1}\', 'MMDDYYYY') AND TO_DATE (\'{date2}\', 'MMDDYYYY')
AND B.CURR_REC_IND = 'Y'
AND B.SRC_REC_DEL_IND = 'N'
AND B.MOST_RCNT_CLM_IND = 'Y'
AND B.RVRSL_IND = 'N'
AND B.ENC_IND = 'N'
AND B.CLM_STS_CD = 'P'
AND B.LNE_STS_CD = 'P'
GROUP BY A.PRSCRB_PROV_LOC_ID) O
ON M.PRSCRB_PROV_LOC_ID = O.PRSCRB_PROV_LOC_ID
GROUP BY O.PRSCRB_PROV_LOC_ID, O.CNT_OFFICE
""".format(date1 = begin_date, date2 = end_date)

engine.execute(pr11_qry)

pr11_df = pd.read_sql('select * from pr11 where rownum < 10', engine)

pr11_df.head()

2020-10-28 23:12:49,064 INFO sqlalchemy.engine.base.Engine 
        BEGIN
           EXECUTE IMMEDIATE 'DROP TABLE ' || 'PR11';
        EXCEPTION
           WHEN OTHERS THEN
              IF SQLCODE != -942 THEN
                 RAISE;
              END IF;
        END;    
    
2020-10-28 23:12:49,065 INFO sqlalchemy.engine.base.Engine {}
2020-10-28 23:12:49,188 INFO sqlalchemy.engine.base.Engine 
CREATE TABLE PR11 AS
SELECT O.PRSCRB_PROV_LOC_ID, O.CNT_OFFICE, COUNT(DISTINCT M.CLM_LNE_FACT_SK) AS CNT_TOT,
(O.CNT_OFFICE/COUNT(DISTINCT M.CLM_LNE_FACT_SK)) AS PR11

--CASE WHEN COUNT(DISTINCT M.CLM_LNE_FACT_SK) < 10 THEN (O.CNT_OFFICE/COUNT(DISTINCT M.CLM_LNE_FACT_SK)) * (COUNT(DISTINCT M.CLM_LNE_FACT_SK) / 10)
--                ELSE  (O.CNT_OFFICE/COUNT(DISTINCT M.CLM_LNE_FACT_SK))
--                END AS PR11


FROM CS_MASTER M
INNER JOIN
(SELECT A.PRSCRB_PROV_LOC_ID, COUNT(DISTINCT A.CLM_LNE_FACT_SK) AS CNT_OFFICE
FROM BIDM_USR_RPTS.CLM_LNE_FACT_V B
INNER JOIN CS_MASTER A
ON B.LNE_FRS

,prscrb_prov_loc_id,cnt_office,cnt_tot,pr11
0,111065,14,38,0.368421
1,171287,493,615,0.801626
2,120939,5,297,0.016835
3,164885,1,170,0.005882
4,106448,2,2,1.000000


# Metric PR12

In [21]:
# Skipping

# Metric PR13

In [22]:
drop_table_if_exists('PR13')

pr13_qry ="""
CREATE TABLE PR13 AS
SELECT 
    C.PRSCRB_PROV_LOC_ID, 
    D.CNT_CLM_D, 
    COUNT(DISTINCT C.ICN_NBR) AS CNT_CLM_P,
    ROUND((D.CNT_CLM_D/(D.CNT_CLM_D + COUNT(DISTINCT C.ICN_NBR))),2) AS PR13
FROM CS_MASTER C
INNER JOIN
(SELECT
  COUNT(DISTINCT A.ICN_NBR) AS CNT_CLM_D,
  A.PRSCRB_PROV_LOC_ID
FROM BIDM_USR_RPTS.CLM_LNE_FACT_V A
INNER JOIN CS_MASTER B
ON A.PRSCRB_PROV_LOC_ID = B.PRSCRB_PROV_LOC_ID
INNER JOIN CDC_WKBK_2018 CDC
ON CDC.NDC = A.NDC_CD 
WHERE CLM_FRST_SVC_DT BETWEEN TO_DATE(\'{date1}\', 'MMDDYYYY') AND TO_DATE(\'{date2}\','MMDDYYYY')
AND A.CLM_PD_DT >= TO_DATE(\'{date1}\', 'MMDDYYYY')
AND A.CLM_TYP_CD IN ('P','Q')
AND A.CURR_REC_IND = 'Y'
AND A.SRC_REC_DEL_IND = 'N'
AND A.RVRSL_IND = 'N'
AND A.ENC_IND = 'N'
AND A.CLM_STS_CD = 'D' -- NORMALLY 'P' FOR PAID
AND SUBSTR(A.ICN_NBR,1,2) = '25'
GROUP BY A.PRSCRB_PROV_LOC_ID) D
ON C.PRSCRB_PROV_LOC_ID = D.PRSCRB_PROV_LOC_ID
GROUP BY C.PRSCRB_PROV_LOC_ID, D.CNT_CLM_D
HAVING COUNT(DISTINCT C.ICN_NBR) > 12
ORDER BY 4 DESC
""".format(date1 = begin_date, date2 = end_date)

engine.execute(pr13_qry)

pr13_df = pd.read_sql('select * from pr13 where rownum < 10', engine)

pr13_df.head()

2020-10-28 23:13:31,168 INFO sqlalchemy.engine.base.Engine 
        BEGIN
           EXECUTE IMMEDIATE 'DROP TABLE ' || 'PR13';
        EXCEPTION
           WHEN OTHERS THEN
              IF SQLCODE != -942 THEN
                 RAISE;
              END IF;
        END;    
    
2020-10-28 23:13:31,169 INFO sqlalchemy.engine.base.Engine {}
2020-10-28 23:13:31,306 INFO sqlalchemy.engine.base.Engine 
CREATE TABLE PR13 AS
SELECT 
    C.PRSCRB_PROV_LOC_ID, 
    D.CNT_CLM_D, 
    COUNT(DISTINCT C.ICN_NBR) AS CNT_CLM_P,
    ROUND((D.CNT_CLM_D/(D.CNT_CLM_D + COUNT(DISTINCT C.ICN_NBR))),2) AS PR13
FROM CS_MASTER C
INNER JOIN
(SELECT
  COUNT(DISTINCT A.ICN_NBR) AS CNT_CLM_D,
  A.PRSCRB_PROV_LOC_ID
FROM BIDM_USR_RPTS.CLM_LNE_FACT_V A
INNER JOIN CS_MASTER B
ON A.PRSCRB_PROV_LOC_ID = B.PRSCRB_PROV_LOC_ID
INNER JOIN CDC_WKBK_2018 CDC
ON CDC.NDC = A.NDC_CD 
WHERE CLM_FRST_SVC_DT BETWEEN TO_DATE('04012020', 'MMDDYYYY') AND TO_DATE('06302020','MMDDYYYY')
AND A.CLM_PD_DT >= TO_DATE('04012020', 'MMDDYYY

,prscrb_prov_loc_id,cnt_clm_d,cnt_clm_p,pr13
0,136987,109,25,0.81
1,165125,55,13,0.81
2,3915,81,20,0.80
3,141069,78,21,0.79
4,121307,66,18,0.79


# Metric PR14
### Skipping

In [23]:
#skipping

# Metric PR15

In [24]:
drop_table_if_exists('PR15')

pr15_qry ="""
CREATE TABLE PR15 AS
SELECT 
PROV.PRSCRB_PROV_LOC_ID, 
COUNT(DISTINCT PROV.CLM_LNE_FACT_SK) AS CNT_CLM
FROM BIDM_USR_RPTS.PROV_LOC_OWN_FACT_V OWN
INNER JOIN
(SELECT M.*, TAX.TAX_NBR_ID AS PRSCRB_PROV_TAX
FROM CS_MASTER M
INNER JOIN
(SELECT DISTINCT A.PRSCRB_PROV_LOC_ID, B.TAX_NBR_ID
FROM CS_MASTER A
INNER JOIN BIDM_USR_RPTS.PROV_LOC_DIM_V B
ON A.PRSCRB_PROV_LOC_ID = B.PROV_LOC_ID
AND B.SRC_REC_DEL_IND = 'N'
AND B.CURR_REC_IND = 'Y') TAX
ON M.PRSCRB_PROV_LOC_ID = TAX.PRSCRB_PROV_LOC_ID) PROV
ON PROV.BILL_PROV_LOC_ID = OWN.PROV_LOC_ID
AND PROV.PRSCRB_PROV_TAX = OWN.PROV_OWN_TAX_NBR_ID
GROUP BY PROV.PRSCRB_PROV_LOC_ID
""".format(date1 = begin_date, date2 = end_date)

engine.execute(pr15_qry)

pr15_df = pd.read_sql('select * from pr15 where rownum < 10', engine)

pr15_df.head()

2020-10-28 23:14:01,027 INFO sqlalchemy.engine.base.Engine 
        BEGIN
           EXECUTE IMMEDIATE 'DROP TABLE ' || 'PR15';
        EXCEPTION
           WHEN OTHERS THEN
              IF SQLCODE != -942 THEN
                 RAISE;
              END IF;
        END;    
    
2020-10-28 23:14:01,028 INFO sqlalchemy.engine.base.Engine {}
2020-10-28 23:14:01,162 INFO sqlalchemy.engine.base.Engine 
CREATE TABLE PR15 AS
SELECT 
PROV.PRSCRB_PROV_LOC_ID, 
COUNT(DISTINCT PROV.CLM_LNE_FACT_SK) AS CNT_CLM
FROM BIDM_USR_RPTS.PROV_LOC_OWN_FACT_V OWN
INNER JOIN
(SELECT M.*, TAX.TAX_NBR_ID AS PRSCRB_PROV_TAX
FROM CS_MASTER M
INNER JOIN
(SELECT DISTINCT A.PRSCRB_PROV_LOC_ID, B.TAX_NBR_ID
FROM CS_MASTER A
INNER JOIN BIDM_USR_RPTS.PROV_LOC_DIM_V B
ON A.PRSCRB_PROV_LOC_ID = B.PROV_LOC_ID
AND B.SRC_REC_DEL_IND = 'N'
AND B.CURR_REC_IND = 'Y') TAX
ON M.PRSCRB_PROV_LOC_ID = TAX.PRSCRB_PROV_LOC_ID) PROV
ON PROV.BILL_PROV_LOC_ID = OWN.PROV_LOC_ID
AND PROV.PRSCRB_PROV_TAX = OWN.PROV_OWN_TAX_NBR_ID
GROUP BY

,prscrb_prov_loc_id,cnt_clm


# Metric PR16

In [25]:
# Skipping

# Metric PR17

In [26]:
# Skipping

# Metric PR18

In [27]:
drop_table_if_exists('PR18')

pr18_qry ="""
CREATE TABLE PR18 AS
WITH CS_PROV_NPI AS
(
SELECT A.PRSCRB_PROV_LOC_ID, B.PROV_NPI_ID
FROM CS_MASTER A
INNER JOIN BIDM_USR_RPTS.PROV_LOC_DIM_V B
ON A.PRSCRB_PROV_LOC_ID = B.PROV_LOC_ID
AND B.CURR_REC_IND = 'Y'
AND B.SRC_REC_DEL_IND = 'N'
)

SELECT DISTINCT C.*, CASE WHEN C.PRSCRB_PROV_LOC_ID LIKE '%' THEN 1
                     END AS PR18
FROM CS_PROV_NPI C
INNER JOIN D98TOGLE.ADVERSE_NPI D -- THIS COMES FROM A ROUTINE DATA MATCHING TASK REFORMED EVERY MONTH.  THERE ARE OTHER POTENTIAL SOURCES THAT WE CAN WORK IN IF NEEDED E.G. DORA LICENSE INFO
ON C.PROV_NPI_ID = D.NPI
""".format(date1 = begin_date, date2 = end_date)

engine.execute(pr18_qry)

pr18_df = pd.read_sql('select * from pr18 where rownum < 10', engine)

pr18_df.head()

2020-10-28 23:14:07,591 INFO sqlalchemy.engine.base.Engine 
        BEGIN
           EXECUTE IMMEDIATE 'DROP TABLE ' || 'PR18';
        EXCEPTION
           WHEN OTHERS THEN
              IF SQLCODE != -942 THEN
                 RAISE;
              END IF;
        END;    
    
2020-10-28 23:14:07,593 INFO sqlalchemy.engine.base.Engine {}
2020-10-28 23:14:07,722 INFO sqlalchemy.engine.base.Engine 
CREATE TABLE PR18 AS
WITH CS_PROV_NPI AS
(
SELECT A.PRSCRB_PROV_LOC_ID, B.PROV_NPI_ID
FROM CS_MASTER A
INNER JOIN BIDM_USR_RPTS.PROV_LOC_DIM_V B
ON A.PRSCRB_PROV_LOC_ID = B.PROV_LOC_ID
AND B.CURR_REC_IND = 'Y'
AND B.SRC_REC_DEL_IND = 'N'
)

SELECT DISTINCT C.*, CASE WHEN C.PRSCRB_PROV_LOC_ID LIKE '%' THEN 1
                     END AS PR18
FROM CS_PROV_NPI C
INNER JOIN D98TOGLE.ADVERSE_NPI D -- THIS COMES FROM A ROUTINE DATA MATCHING TASK REFORMED EVERY MONTH.  THERE ARE OTHER POTENTIAL SOURCES THAT WE CAN WORK IN IF NEEDED E.G. DORA LICENSE INFO
ON C.PROV_NPI_ID = D.NPI

2020-10-28 23:14:07

,prscrb_prov_loc_id,prov_npi_id,pr18
0,8480,1285769158,1
1,121062,1720200892,1
2,110605,1700064391,1
3,17347,1588626394,1
4,7224,1245265131,1


# Metric PR19

In [28]:
drop_table_if_exists('PR19')

pr19_qry ="""
CREATE TABLE PR19 AS
WITH OP_RX AS
(
SELECT A.PRSCRB_PROV_LOC_ID, A.CLM_LNE_FACT_SK, A.MCAID_ID, A.PRSRB_DT, A.DRUG_DSPN_DT
FROM CS_MASTER A
WHERE A."Class" = 'Opioid'
AND A.GENNME NOT IN ('Buprenorphine/naloxone')
)
,

SUB_RX AS
(
SELECT A.PRSCRB_PROV_LOC_ID, A.CLM_LNE_FACT_SK, A.MCAID_ID, A.PRSRB_DT, A.DRUG_DSPN_DT
FROM CS_MASTER A
WHERE A.GENNME IN ('Buprenorphine/naloxone')
)

SELECT M.PRSCRB_PROV_LOC_ID, X.CNT_MEM_SUB, COUNT(DISTINCT M.MCAID_ID) AS CNT_MEM_TOT, 
(X.CNT_MEM_SUB/COUNT(DISTINCT M.MCAID_ID)) AS PR19
--X.CNT_MEM_SUB as PR19

FROM CS_MASTER M
INNER JOIN
(SELECT A.PRSCRB_PROV_LOC_ID, COUNT(DISTINCT A.MCAID_ID) AS CNT_MEM_SUB
FROM OP_RX A
INNER JOIN SUB_RX B
ON A.MCAID_ID = B.MCAID_ID
AND A.DRUG_DSPN_DT BETWEEN (B.DRUG_DSPN_DT - 7) AND (B.DRUG_DSPN_DT +7)
GROUP BY A.PRSCRB_PROV_LOC_ID) X
ON M.PRSCRB_PROV_LOC_ID = X.PRSCRB_PROV_LOC_ID
GROUP BY M.PRSCRB_PROV_LOC_ID, X.CNT_MEM_SUB
""".format(date1 = begin_date, date2 = end_date)

engine.execute(pr19_qry)

pr19_df = pd.read_sql('select * from pr19 where rownum < 10', engine)

pr19_df.head()

2020-10-28 23:14:08,348 INFO sqlalchemy.engine.base.Engine 
        BEGIN
           EXECUTE IMMEDIATE 'DROP TABLE ' || 'PR19';
        EXCEPTION
           WHEN OTHERS THEN
              IF SQLCODE != -942 THEN
                 RAISE;
              END IF;
        END;    
    
2020-10-28 23:14:08,350 INFO sqlalchemy.engine.base.Engine {}
2020-10-28 23:14:08,456 INFO sqlalchemy.engine.base.Engine 
CREATE TABLE PR19 AS
WITH OP_RX AS
(
SELECT A.PRSCRB_PROV_LOC_ID, A.CLM_LNE_FACT_SK, A.MCAID_ID, A.PRSRB_DT, A.DRUG_DSPN_DT
FROM CS_MASTER A
WHERE A."Class" = 'Opioid'
AND A.GENNME NOT IN ('Buprenorphine/naloxone')
)
,

SUB_RX AS
(
SELECT A.PRSCRB_PROV_LOC_ID, A.CLM_LNE_FACT_SK, A.MCAID_ID, A.PRSRB_DT, A.DRUG_DSPN_DT
FROM CS_MASTER A
WHERE A.GENNME IN ('Buprenorphine/naloxone')
)

SELECT M.PRSCRB_PROV_LOC_ID, X.CNT_MEM_SUB, COUNT(DISTINCT M.MCAID_ID) AS CNT_MEM_TOT, 
(X.CNT_MEM_SUB/COUNT(DISTINCT M.MCAID_ID)) AS PR19
--X.CNT_MEM_SUB as PR19

FROM CS_MASTER M
INNER JOIN
(SELECT A.PRSCRB_PROV_

,prscrb_prov_loc_id,cnt_mem_sub,cnt_mem_tot,pr19
0,180425,1,7,0.142857
1,171420,1,22,0.045455
2,116654,1,105,0.009524
3,19384,1,71,0.014085
4,149244,1,4,0.250000


# Metric PR20

In [29]:
#skipping

# Metric PR21

In [30]:
drop_table_if_exists('PR21')

p21_qry ="""
CREATE TABLE PR21 AS

WITH ATYP_RX AS
(
SELECT A.PRSCRB_PROV_LOC_ID, A.CLM_LNE_FACT_SK, A.MCAID_ID, A.PRSRB_DT, A.DRUG_DSPN_DT
FROM BIDM_USR_RPTS.CLM_LNE_FACT_V A
INNER JOIN CS_MASTER B
ON A.PRSCRB_PROV_LOC_ID = B.PRSCRB_PROV_LOC_ID
WHERE A.NDC_CD IN ( SELECT DISTINCT
  B.NDC_CD
FROM BIDM_USR_RPTS.CLM_LNE_FACT_V B
WHERE b.HIC3_THRPTC_DRUG_CLS_CD IN ('H7X', 'H7T')
AND B.CURR_REC_IND = 'Y'
AND B.SRC_REC_DEL_IND = 'N'
AND B.MOST_RCNT_CLM_IND = 'Y'
AND B.RVRSL_IND = 'N'))
,

OP_RX AS
(
SELECT A.PRSCRB_PROV_LOC_ID, A.CLM_LNE_FACT_SK, A.MCAID_ID, A.PRSRB_DT, A.DRUG_DSPN_DT
FROM CS_MASTER A
WHERE A."Class" = 'Opioid'
)


SELECT M.PRSCRB_PROV_LOC_ID, X.CNT_MEM_ATYP, COUNT(DISTINCT M.MCAID_ID) AS CNT_MEM_TOT, 
(X.CNT_MEM_ATYP/COUNT(DISTINCT M.MCAID_ID)) AS PR21
--(X.CNT_MEM_ATYP) AS PR21
FROM CS_MASTER M
INNER JOIN
(SELECT A.PRSCRB_PROV_LOC_ID, COUNT(DISTINCT A.MCAID_ID) AS CNT_MEM_ATYP
FROM OP_RX A
INNER JOIN ATYP_RX B
ON A.MCAID_ID = B.MCAID_ID
AND A.DRUG_DSPN_DT BETWEEN (B.DRUG_DSPN_DT - 7) AND (B.DRUG_DSPN_DT +7)
GROUP BY A.PRSCRB_PROV_LOC_ID) X
ON M.PRSCRB_PROV_LOC_ID = X.PRSCRB_PROV_LOC_ID
GROUP BY M.PRSCRB_PROV_LOC_ID, X.CNT_MEM_ATYP
""".format(date1 = begin_date, date2 = end_date)

engine.execute(p21_qry)

p21_df = pd.read_sql('select * from PR21 where rownum < 10', engine)

p21_df.head()

2020-10-28 23:14:08,947 INFO sqlalchemy.engine.base.Engine 
        BEGIN
           EXECUTE IMMEDIATE 'DROP TABLE ' || 'PR21';
        EXCEPTION
           WHEN OTHERS THEN
              IF SQLCODE != -942 THEN
                 RAISE;
              END IF;
        END;    
    
2020-10-28 23:14:08,948 INFO sqlalchemy.engine.base.Engine {}
2020-10-28 23:14:09,056 INFO sqlalchemy.engine.base.Engine 
CREATE TABLE PR21 AS

WITH ATYP_RX AS
(
SELECT A.PRSCRB_PROV_LOC_ID, A.CLM_LNE_FACT_SK, A.MCAID_ID, A.PRSRB_DT, A.DRUG_DSPN_DT
FROM BIDM_USR_RPTS.CLM_LNE_FACT_V A
INNER JOIN CS_MASTER B
ON A.PRSCRB_PROV_LOC_ID = B.PRSCRB_PROV_LOC_ID
WHERE A.NDC_CD IN ( SELECT DISTINCT
  B.NDC_CD
FROM BIDM_USR_RPTS.CLM_LNE_FACT_V B
WHERE b.HIC3_THRPTC_DRUG_CLS_CD IN ('H7X', 'H7T')
AND B.CURR_REC_IND = 'Y'
AND B.SRC_REC_DEL_IND = 'N'
AND B.MOST_RCNT_CLM_IND = 'Y'
AND B.RVRSL_IND = 'N'))
,

OP_RX AS
(
SELECT A.PRSCRB_PROV_LOC_ID, A.CLM_LNE_FACT_SK, A.MCAID_ID, A.PRSRB_DT, A.DRUG_DSPN_DT
FROM CS_MASTER A
WHERE A

,prscrb_prov_loc_id,cnt_mem_atyp,cnt_mem_tot,pr21
0,130656,3,118,0.025424
1,121692,5,96,0.052083
2,171798,1,6,0.166667
3,135472,2,16,0.125000
4,111273,1,14,0.071429


# Metric PR22
### Driving distance

In [31]:
drop_table_if_exists('PROV_CLNT_ADDR')

p22_qry ="""
CREATE TABLE PROV_CLNT_ADDR AS
WITH PROV_CLNT_ADDR_1 AS
(
SELECT distinct A.CLM_LNE_FACT_SK, A.PRSCRB_PROV_LOC_ID, B.PRSCRB_PROV_LOC_DIM_SK,
A.MCAID_ID, B.CLNT_DIM_SK
from CS_MASTER A
INNER JOIN BIDM_USR_RPTS.CLM_LNE_FACT_V B
ON B.CLM_LNE_FACT_SK = A.CLM_LNE_FACT_SK
)
,
PROV_CLNT_ADDR_2 AS
(
SELECT A.*, B.PROV_LOC_ID, B.SVC_ADDR_LNE_1_TX, B.SVC_ADDR_LNE_2_TX, B.SVC_ADDR_LNE_3_TX, B.SVC_CTY_NM, B.SVC_ST_CD, B.SVC_PSTL_CD, B.SVC_ZIP_PLS_4_CD, B.SVC_ADDR_LAT_NBR, B.SVC_ADDR_LONG_NBR
FROM PROV_CLNT_ADDR_1 A
INNER JOIN BIDM_USR_RPTS.PROV_LOC_DIM_V B
ON A.PRSCRB_PROV_LOC_DIM_SK = B.PROV_LOC_DIM_SK
)

SELECT A.*, B.MCAID_ID AS MCAID_ID_2, B.HOME_ADDR_LNE_1_TX, B.HOME_ADDR_LNE_2_TX, B.HOME_ADDR_LNE_3_TX, B.HOME_CTY_NM, B.HOME_ST_CD, B.HOME_PSTL_CD, B.HOME_ZIP_PLS_4_CD, B.HOME_ADDR_LAT_NBR, B.HOME_ADDR_LONG_NBR
FROM PROV_CLNT_ADDR_2 A
INNER JOIN BIDM_USR_RPTS.CLNT_DIM_V B
ON A.CLNT_DIM_SK = B.CLNT_DIM_SK
""".format(date1 = begin_date, date2 = end_date)

engine.execute(p22_qry)

p22_df_step1 = pd.read_sql('select * from PROV_CLNT_ADDR', engine)

p22_df_step1.head()


2020-10-28 23:15:21,298 INFO sqlalchemy.engine.base.Engine 
        BEGIN
           EXECUTE IMMEDIATE 'DROP TABLE ' || 'PROV_CLNT_ADDR';
        EXCEPTION
           WHEN OTHERS THEN
              IF SQLCODE != -942 THEN
                 RAISE;
              END IF;
        END;    
    
2020-10-28 23:15:21,300 INFO sqlalchemy.engine.base.Engine {}
2020-10-28 23:15:21,462 INFO sqlalchemy.engine.base.Engine 
CREATE TABLE PROV_CLNT_ADDR AS
WITH PROV_CLNT_ADDR_1 AS
(
SELECT distinct A.CLM_LNE_FACT_SK, A.PRSCRB_PROV_LOC_ID, B.PRSCRB_PROV_LOC_DIM_SK,
A.MCAID_ID, B.CLNT_DIM_SK
from CS_MASTER A
INNER JOIN BIDM_USR_RPTS.CLM_LNE_FACT_V B
ON B.CLM_LNE_FACT_SK = A.CLM_LNE_FACT_SK
)
,
PROV_CLNT_ADDR_2 AS
(
SELECT A.*, B.PROV_LOC_ID, B.SVC_ADDR_LNE_1_TX, B.SVC_ADDR_LNE_2_TX, B.SVC_ADDR_LNE_3_TX, B.SVC_CTY_NM, B.SVC_ST_CD, B.SVC_PSTL_CD, B.SVC_ZIP_PLS_4_CD, B.SVC_ADDR_LAT_NBR, B.SVC_ADDR_LONG_NBR
FROM PROV_CLNT_ADDR_1 A
INNER JOIN BIDM_USR_RPTS.PROV_LOC_DIM_V B
ON A.PRSCRB_PROV_LOC_DIM_SK = B.PROV_

,clm_lne_fact_sk,prscrb_prov_loc_id,prscrb_prov_loc_dim_sk,mcaid_id,clnt_dim_sk,prov_loc_id,svc_addr_lne_1_tx,svc_addr_lne_2_tx,svc_addr_lne_3_tx,svc_cty_nm,svc_st_cd,svc_pstl_cd,svc_zip_pls_4_cd,svc_addr_lat_nbr,svc_addr_long_nbr,mcaid_id_2,home_addr_lne_1_tx,home_addr_lne_2_tx,home_addr_lne_3_tx,home_cty_nm,home_st_cd,home_pstl_cd,home_zip_pls_4_cd,home_addr_lat_nbr,home_addr_long_nbr
0,916657921,135426,489040,O073195,48914422,135426,777 BANNOCK ST,None,None,DENVER,CO,80204,4507,39.728140,-104.990176,O073195,4121 S DUNKIRK WAY,None,None,AURORA,CO,80013,5114,39.642289,-104.764196
1,888753563,180475,2740591,G280495,58711988,180475,5911 MIDDLEFIELD RD,None,None,LITTLETON,CO,80123,4289,39.609292,-105.035004,G280495,GENERAL DELIVERY,None,None,COLORADO SPRINGS,CO,80903,9999,38.833430,-104.821810
2,904602072,179784,2636283,G669439,61078400,179784,10375 PARK MEADOWS DR,SUITE 270,None,LONE TREE,CO,80124,6735,39.540987,-104.871283,G669439,7980 W HOOVER PL,None,None,LITTLETON,CO,80123,3528,39.597576,-105.085136
3,904538270,179784,2636283,G669439,61078400,179784,10375 PARK MEADOWS DR,SUITE 270,None,LONE TREE,CO,80124,6735,39.540987,-104.871283,G669439,7980 W HOOVER PL,None,None,LITTLETON,CO,80123,3528,39.597576,-105.085136
4,916559965,179784,2636283,P865459,67785462,179784,10375 PARK MEADOWS DR,SUITE 270,None,LONE TREE,CO,80124,6735,39.540987,-104.871283,P865459,9006 S MICA MINE GULCH RD,TRLR 132,None,LITTLETON,CO,80127,0000,39.552891,-105.200971


## Function to compute distance

In [32]:
from math import sin, cos, sqrt, atan2, radians
def get_distance(point1, point2):
    R = 3958.756
    lat1 = radians(point1[0])  #insert value
    lon1 = radians(point1[1])
    lat2 = radians(point2[0])
    lon2 = radians(point2[1])

    dlon = lon2 - lon1
    dlat = lat2- lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))
    distance = R * c
    return distance

def get_dists(df):
    dist = []
    for i in range(df.shape[0]):
        if (i%10000) == 0:
            print(i)
        dist.append(get_distance([df.iloc[[i]].home_addr_lat_nbr[i], df.iloc[[i]].home_addr_long_nbr[i]],[df.iloc[[i]].svc_addr_lat_nbr[i], df.iloc[[i]].svc_addr_long_nbr[i]]))  
    return (pd.Series(dist)) 

In [33]:
p22_df_step2 = p22_df_step1
p22_df_step2['dist'] = get_dists(p22_df_step1)
p22_df_step2.loc[p22_df_step2.home_addr_lat_nbr==0, 'dist'] = 0
p22_df_step2['dist_over50'] = p22_df_step2['dist'] > 50
p22_df_step2.loc[p22_df_step2['dist'] > 300, 'dist_over50'] = False


p22_df_step2.head() 

0
10000
20000
30000
40000
50000
60000
70000
80000
90000
100000
110000
120000
130000
140000
150000
160000
170000
180000
190000
200000
210000
220000


,clm_lne_fact_sk,prscrb_prov_loc_id,prscrb_prov_loc_dim_sk,mcaid_id,clnt_dim_sk,prov_loc_id,svc_addr_lne_1_tx,svc_addr_lne_2_tx,svc_addr_lne_3_tx,svc_cty_nm,svc_st_cd,svc_pstl_cd,svc_zip_pls_4_cd,svc_addr_lat_nbr,svc_addr_long_nbr,mcaid_id_2,home_addr_lne_1_tx,home_addr_lne_2_tx,home_addr_lne_3_tx,home_cty_nm,home_st_cd,home_pstl_cd,home_zip_pls_4_cd,home_addr_lat_nbr,home_addr_long_nbr,dist,dist_over50
0,916657921,135426,489040,O073195,48914422,135426,777 BANNOCK ST,None,None,DENVER,CO,80204,4507,39.728140,-104.990176,O073195,4121 S DUNKIRK WAY,None,None,AURORA,CO,80013,5114,39.642289,-104.764196,13.400136,False
1,888753563,180475,2740591,G280495,58711988,180475,5911 MIDDLEFIELD RD,None,None,LITTLETON,CO,80123,4289,39.609292,-105.035004,G280495,GENERAL DELIVERY,None,None,COLORADO SPRINGS,CO,80903,9999,38.833430,-104.821810,54.808017,True
2,904602072,179784,2636283,G669439,61078400,179784,10375 PARK MEADOWS DR,SUITE 270,None,LONE TREE,CO,80124,6735,39.540987,-104.871283,G669439,7980 W HOOVER PL,None,None,LITTLETON,CO,80123,3528,39.597576,-105.085136,12.042413,False
3,904538270,179784,2636283,G669439,61078400,179784,10375 PARK MEADOWS DR,SUITE 270,None,LONE TREE,CO,80124,6735,39.540987,-104.871283,G669439,7980 W HOOVER PL,None,None,LITTLETON,CO,80123,3528,39.597576,-105.085136,12.042413,False
4,916559965,179784,2636283,P865459,67785462,179784,10375 PARK MEADOWS DR,SUITE 270,None,LONE TREE,CO,80124,6735,39.540987,-104.871283,P865459,9006 S MICA MINE GULCH RD,TRLR 132,None,LITTLETON,CO,80127,0000,39.552891,-105.200971,17.584382,False


In [34]:
# p22_df_step3 = p22_df_step2. \
#     groupby(['prscrb_prov_loc_id'], as_index=False). \
#     agg({'dist_over50': 'sum'}). \
#     rename(columns={"dist_over50": "pr22"})  
 
# p22_df_step3.head()  

p22_df_step3 = p22_df_step2. \
    groupby(['prscrb_prov_loc_id'], as_index=False). \
    agg({'dist': 'mean'}). \
    rename(columns={"dist": "pr22"})  
 
p22_df_step3.head()  

,prscrb_prov_loc_id,pr22
0,100014,14.926587
1,100016,28.352538
2,100017,27.956378
3,100018,17.286081
4,100023,20.034550


In [35]:
# Write back to DB
drop_table_if_exists('PR22')
p22_df_step3.to_sql('PR22', engine, if_exists='replace', index=False, dtype={"prscrb_prov_loc_id": types.VARCHAR(256)})

p22_df = pd.read_sql('select * from PR22 where rownum < 10', engine)

p22_df.head()

2020-10-28 23:27:00,099 INFO sqlalchemy.engine.base.Engine 
        BEGIN
           EXECUTE IMMEDIATE 'DROP TABLE ' || 'PR22';
        EXCEPTION
           WHEN OTHERS THEN
              IF SQLCODE != -942 THEN
                 RAISE;
              END IF;
        END;    
    
2020-10-28 23:27:00,100 INFO sqlalchemy.engine.base.Engine {}
2020-10-28 23:27:00,258 INFO sqlalchemy.engine.base.Engine SELECT table_name FROM all_tables WHERE table_name = :name AND owner = :schema_name
2020-10-28 23:27:00,260 INFO sqlalchemy.engine.base.Engine {'name': 'PR22', 'schema_name': 'U4J8175'}
2020-10-28 23:27:00,345 INFO sqlalchemy.engine.base.Engine 
CREATE TABLE "PR22" (
	prscrb_prov_loc_id VARCHAR(256 CHAR), 
	pr22 FLOAT
)


2020-10-28 23:27:00,347 INFO sqlalchemy.engine.base.Engine {}
2020-10-28 23:27:00,410 INFO sqlalchemy.engine.base.Engine COMMIT
2020-10-28 23:27:00,500 INFO sqlalchemy.engine.base.Engine BEGIN (implicit)
2020-10-28 23:27:00,637 INFO sqlalchemy.engine.base.Engine INSERT INTO 

/opt/conda/envs/Python-3.6-WMLCE/lib/python3.6/site-packages/pandas/io/sql.py:1191: UserWarning: The provided table name 'PR22' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  warnings.warn(msg, UserWarning)


,prscrb_prov_loc_id,pr22
0,105691,6.439075
1,105693,1.758067
2,105699,13.271845
3,105700,6.675857
4,105707,17.707574


# Metric PR23

In [36]:
drop_table_if_exists('PR23')

p23_qry ="""
CREATE TABLE PR23 AS
SELECT M.PRSCRB_PROV_LOC_ID, COUNT(M.CLM_LNE_FACT_SK) AS CNT_LNE, E.CNT_CLM_MINOR,
(E.CNT_CLM_MINOR/ COUNT(M.CLM_LNE_FACT_SK)) AS PR23

--CASE WHEN COUNT(DISTINCT M.MCAID_ID) < 10 THEN (E.CNT_CLM_MINOR/ COUNT(M.CLM_LNE_FACT_SK)) * (COUNT(M.CLM_LNE_FACT_SK) / 10)
--                ELSE (E.CNT_CLM_MINOR/ COUNT(M.CLM_LNE_FACT_SK))
--                END AS PR23



FROM CS_MASTER M
INNER JOIN
(SELECT A.PRSCRB_PROV_LOC_ID, COUNT(A.CLM_LNE_FACT_SK) AS CNT_CLM_MINOR
FROM CS_MASTER A
WHERE A.CLNT_AGE_QTY < 18 --THIS FIELD IS (FDOS - DOB)
AND A."Class" = 'Opioid'
GROUP BY A.PRSCRB_PROV_LOC_ID) E
ON M.PRSCRB_PROV_LOC_ID = E.PRSCRB_PROV_LOC_ID
and M."Class" = 'Opioid'
GROUP BY M.PRSCRB_PROV_LOC_ID, E.CNT_CLM_MINOR
""".format(date1 = begin_date, date2 = end_date)

engine.execute(p23_qry)

p23_df = pd.read_sql('select * from PR23 where rownum < 10', engine)

p23_df.head()

2020-10-28 23:27:02,101 INFO sqlalchemy.engine.base.Engine 
        BEGIN
           EXECUTE IMMEDIATE 'DROP TABLE ' || 'PR23';
        EXCEPTION
           WHEN OTHERS THEN
              IF SQLCODE != -942 THEN
                 RAISE;
              END IF;
        END;    
    
2020-10-28 23:27:02,103 INFO sqlalchemy.engine.base.Engine {}
2020-10-28 23:27:02,211 INFO sqlalchemy.engine.base.Engine 
CREATE TABLE PR23 AS
SELECT M.PRSCRB_PROV_LOC_ID, COUNT(M.CLM_LNE_FACT_SK) AS CNT_LNE, E.CNT_CLM_MINOR,
(E.CNT_CLM_MINOR/ COUNT(M.CLM_LNE_FACT_SK)) AS PR23

--CASE WHEN COUNT(DISTINCT M.MCAID_ID) < 10 THEN (E.CNT_CLM_MINOR/ COUNT(M.CLM_LNE_FACT_SK)) * (COUNT(M.CLM_LNE_FACT_SK) / 10)
--                ELSE (E.CNT_CLM_MINOR/ COUNT(M.CLM_LNE_FACT_SK))
--                END AS PR23



FROM CS_MASTER M
INNER JOIN
(SELECT A.PRSCRB_PROV_LOC_ID, COUNT(A.CLM_LNE_FACT_SK) AS CNT_CLM_MINOR
FROM CS_MASTER A
WHERE A.CLNT_AGE_QTY < 18 --THIS FIELD IS (FDOS - DOB)
AND A."Class" = 'Opioid'
GROUP BY A.PRSCRB

,prscrb_prov_loc_id,cnt_lne,cnt_clm_minor,pr23
0,145105,33,1,0.030303
1,124984,63,1,0.015873
2,139276,30,3,0.100000
3,119830,62,1,0.016129
4,126137,160,21,0.131250


# Metric PR24

In [37]:
drop_table_if_exists('PR24')

p24_qry ="""
CREATE TABLE PR24 AS
WITH SUBUTEX_RX AS
(
SELECT A.PRSCRB_PROV_LOC_ID, A.CLM_LNE_FACT_SK, A.MCAID_ID, A.PRSRB_DT, A.DRUG_DSPN_DT
FROM CS_MASTER A
WHERE A.GENNME IN ('Buprenorphine Hydrochloride')
)

SELECT M.PRSCRB_PROV_LOC_ID, X.CNT_MEM_SUBUTEX, COUNT(DISTINCT M.MCAID_ID) AS CNT_MEM_TOT, 
(X.CNT_MEM_SUBUTEX/COUNT(DISTINCT M.MCAID_ID)) AS PR24

--CASE WHEN COUNT(DISTINCT M.MCAID_ID) < 10 THEN (X.CNT_MEM_SUBUTEX/COUNT(DISTINCT M.MCAID_ID)) * (COUNT(DISTINCT M.MCAID_ID) / 10)
--                ELSE (X.CNT_MEM_SUBUTEX/COUNT(DISTINCT M.MCAID_ID))
--                END AS PR24


FROM CS_MASTER M
INNER JOIN
(SELECT A.PRSCRB_PROV_LOC_ID, COUNT(DISTINCT A.MCAID_ID) AS CNT_MEM_SUBUTEX
FROM SUBUTEX_RX A
GROUP BY A.PRSCRB_PROV_LOC_ID) X
ON M.PRSCRB_PROV_LOC_ID = X.PRSCRB_PROV_LOC_ID
GROUP BY M.PRSCRB_PROV_LOC_ID, X.CNT_MEM_SUBUTEX
""".format(date1 = begin_date, date2 = end_date)

engine.execute(p24_qry)

p24_df = pd.read_sql('select * from PR24 where pr24 > .3', engine)

p24_df.head()

2020-10-28 23:27:02,864 INFO sqlalchemy.engine.base.Engine 
        BEGIN
           EXECUTE IMMEDIATE 'DROP TABLE ' || 'PR24';
        EXCEPTION
           WHEN OTHERS THEN
              IF SQLCODE != -942 THEN
                 RAISE;
              END IF;
        END;    
    
2020-10-28 23:27:02,865 INFO sqlalchemy.engine.base.Engine {}
2020-10-28 23:27:02,971 INFO sqlalchemy.engine.base.Engine 
CREATE TABLE PR24 AS
WITH SUBUTEX_RX AS
(
SELECT A.PRSCRB_PROV_LOC_ID, A.CLM_LNE_FACT_SK, A.MCAID_ID, A.PRSRB_DT, A.DRUG_DSPN_DT
FROM CS_MASTER A
WHERE A.GENNME IN ('Buprenorphine Hydrochloride')
)

SELECT M.PRSCRB_PROV_LOC_ID, X.CNT_MEM_SUBUTEX, COUNT(DISTINCT M.MCAID_ID) AS CNT_MEM_TOT, 
(X.CNT_MEM_SUBUTEX/COUNT(DISTINCT M.MCAID_ID)) AS PR24

--CASE WHEN COUNT(DISTINCT M.MCAID_ID) < 10 THEN (X.CNT_MEM_SUBUTEX/COUNT(DISTINCT M.MCAID_ID)) * (COUNT(DISTINCT M.MCAID_ID) / 10)
--                ELSE (X.CNT_MEM_SUBUTEX/COUNT(DISTINCT M.MCAID_ID))
--                END AS PR24


FROM CS_MASTER M


,prscrb_prov_loc_id,cnt_mem_subutex,cnt_mem_tot,pr24
0,110343,3,6,0.500000
1,119372,10,30,0.333333
2,174251,1,1,1.000000
3,162692,4,12,0.333333
4,105543,1,1,1.000000


# Metric PR25

In [38]:
drop_table_if_exists('PR25')

p25_qry ="""
CREATE TABLE PR25 AS
WITH GABA_RX AS
(
SELECT A.PRSCRB_PROV_LOC_ID, A.CLM_LNE_FACT_SK, A.MCAID_ID, A.PRSRB_DT, A.DRUG_DSPN_DT
FROM BIDM_USR_RPTS.CLM_LNE_FACT_V A
INNER JOIN CS_MASTER B
ON A.PRSCRB_PROV_LOC_ID = B.PRSCRB_PROV_LOC_ID
WHERE A.NDC_CD IN ('00093444301',
'00093444305',
'00093444310',
'00093444401',
'00093444405',
'00228263611',
'00228263650',
'00228263711',
'00228263750',
'00228266511',
'00228266550',
'00228266611',
'00228266650',
'00228266711',
'00228266750',
'00378542705',
'00904563161',
'00904563189',
'00904563261',
'00904666561',
'00904666661',
'14550051104',
'14550051202',
'14550051204',
'14550051302',
'14550051304',
'16714033001',
'16714033002',
'16714033201',
'16714033202',
'16714050301',
'16714050302',
'16714050401',
'16714050402',
'16714050501',
'16714050502',
'16714066101',
'16714066102',
'16714066201',
'16714066202',
'16714066301',
'16714066302',
'31722022101',
'31722022105',
'31722022201',
'31722022205',
'31722022301',
'31722022305',
'31722040501',
'31722040505',
'31722040601',
'31722040605',
'38779198005',
'38779246102',
'38779246104',
'38779246105',
'38779246108',
'38779246109',
'42192060816',
'43547026550',
'43547026650',
'43547026750',
'43547033210',
'43547033250',
'43547033310',
'43547033350',
'43547038310',
'43547038350',
'43547038410',
'43547038450',
'43547038510',
'43547038550',
'43547038910',
'43547038950',
'43547039010',
'43547039050',
'45963055511',
'45963055550',
'45963055611',
'45963055650',
'45963055711',
'45963055750',
'49483060501',
'49483060550',
'49483060601',
'49483060650',
'49483060701',
'49483060750',
'50228017705',
'50228017801',
'50228017805',
'50228017901',
'50228017905',
'50228018001',
'50228018005',
'50228018101',
'50228018105',
'50383031107',
'50383031109',
'50383031147',
'51224002160',
'51552090204',
'51927421300',
'52372091202',
'53746010101',
'53746010105',
'53746010110',
'53746010201',
'53746010205',
'53746010210',
'53746010301',
'53746010305',
'58657062001',
'58657062050',
'58657062101',
'58657062150',
'58657062201',
'58657062250',
'58657062301',
'58657062350',
'58657062401',
'58657062450',
'59762502301',
'59762502401',
'59762502501',
'59762502701',
'59762502801',
'59762505001',
'60505011200',
'60505011201',
'60505011208',
'60505011301',
'60505011308',
'60505011401',
'60505011405',
'60505255101',
'60505255105',
'60505255201',
'60505255205',
'62756013702',
'62756013705',
'62756013802',
'62756013805',
'62756013905',
'62756020201',
'62756020203',
'62756020401',
'62756020403',
'62991220401',
'62991220402',
'62991220403',
'62991220408',
'63275997005',
'63739023610',
'63739037510',
'63739056010',
'63739059110',
'63739067510',
'63739068910',
'63739098410',
'64380086807',
'65162010110',
'65162010111',
'65162010150',
'65162010210',
'65162010211',
'65162010250',
'65162010310',
'65162010350',
'65162069890',
'65862019801',
'65862019805',
'65862019899',
'65862019901',
'65862019905',
'65862019999',
'65862020001',
'65862020005',
'65862052301',
'65862052305',
'65862052401',
'65862052405',
'67877022201',
'67877022205',
'67877022210',
'67877022301',
'67877022305',
'67877022310',
'67877022401',
'67877022405',
'67877022410',
'67877042801',
'67877042805',
'67877042901',
'67877042905',
'68001000600',
'68001000603',
'68001000700',
'68001000703',
'68001041100',
'68001041103',
'68001041200',
'68001041203',
'68084076201',
'68084077401',
'68084078301',
'68382020401',
'68382020405',
'68382020501',
'68382020505',
'68462012601',
'68462012605',
'68462012701',
'68462012705',
'69097081107',
'69097081112',
'69097081207',
'69097081212',
'69097081307',
'69097081312',
'69097081407',
'69097081412',
'69097081507',
'69097081512',
'69097094307',
'69097094312',
'69367013104',
'69367013106',
'69367013206',
'69367013304',
'69367013306',
'69367013404',
'69367013406',
'69367013504',
'69367013506',
'71093011104',
'71093011105',
'71093011204',
'71093011205',
'71093012005',
'71093012105',
'71093012204',
'71093012205',
'71399055605',
'71717010210',
'71717010250',
'71717010310',
'71717010350',
'76282032305',
'76282040501',
'76282040505',
'76282040590',
'76282040601',
'76282040605',
'76282062705'))
,

OP_RX AS
(
SELECT A.PRSCRB_PROV_LOC_ID, A.CLM_LNE_FACT_SK, A.MCAID_ID, A.PRSRB_DT, A.DRUG_DSPN_DT
FROM CS_MASTER A
WHERE A."Class" = 'Opioid'
)


SELECT M.PRSCRB_PROV_LOC_ID, X.CNT_MEM_GABA, COUNT(DISTINCT M.MCAID_ID) AS CNT_MEM_TOT, 
(X.CNT_MEM_GABA/COUNT(DISTINCT M.MCAID_ID)) AS PR25

--CASE WHEN COUNT(DISTINCT M.MCAID_ID) < 10 THEN (X.CNT_MEM_GABA/COUNT(DISTINCT M.MCAID_ID)) * (COUNT(DISTINCT M.MCAID_ID) / 10)
--                ELSE (X.CNT_MEM_GABA/COUNT(DISTINCT M.MCAID_ID))
--                END AS PR25


FROM CS_MASTER M
INNER JOIN
(SELECT A.PRSCRB_PROV_LOC_ID, COUNT(DISTINCT A.MCAID_ID) AS CNT_MEM_GABA
FROM OP_RX A
INNER JOIN GABA_RX B
ON A.MCAID_ID = B.MCAID_ID
AND A.DRUG_DSPN_DT BETWEEN (B.DRUG_DSPN_DT - 7) AND (B.DRUG_DSPN_DT +7)
GROUP BY A.PRSCRB_PROV_LOC_ID) X
ON M.PRSCRB_PROV_LOC_ID = X.PRSCRB_PROV_LOC_ID
GROUP BY M.PRSCRB_PROV_LOC_ID, X.CNT_MEM_GABA
""".format(date1 = begin_date, date2 = end_date)

engine.execute(p25_qry)

p25_df = pd.read_sql('select * from PR25 where rownum < 10', engine)

p25_df.head()

2020-10-28 23:27:04,048 INFO sqlalchemy.engine.base.Engine 
        BEGIN
           EXECUTE IMMEDIATE 'DROP TABLE ' || 'PR25';
        EXCEPTION
           WHEN OTHERS THEN
              IF SQLCODE != -942 THEN
                 RAISE;
              END IF;
        END;    
    
2020-10-28 23:27:04,050 INFO sqlalchemy.engine.base.Engine {}
2020-10-28 23:27:04,159 INFO sqlalchemy.engine.base.Engine 
CREATE TABLE PR25 AS
WITH GABA_RX AS
(
SELECT A.PRSCRB_PROV_LOC_ID, A.CLM_LNE_FACT_SK, A.MCAID_ID, A.PRSRB_DT, A.DRUG_DSPN_DT
FROM BIDM_USR_RPTS.CLM_LNE_FACT_V A
INNER JOIN CS_MASTER B
ON A.PRSCRB_PROV_LOC_ID = B.PRSCRB_PROV_LOC_ID
WHERE A.NDC_CD IN ('00093444301',
'00093444305',
'00093444310',
'00093444401',
'00093444405',
'00228263611',
'00228263650',
'00228263711',
'00228263750',
'00228266511',
'00228266550',
'00228266611',
'00228266650',
'00228266711',
'00228266750',
'00378542705',
'00904563161',
'00904563189',
'00904563261',
'00904666561',
'00904666661',
'14550051104',
'14550051202',
'1

,prscrb_prov_loc_id,cnt_mem_gaba,cnt_mem_tot,pr25
0,120780,1,12,0.083333
1,139358,3,46,0.065217
2,153504,10,80,0.125000
3,126141,1,37,0.027027
4,122660,1,29,0.034483


In [39]:
drop_table_if_exists('PR26')

p26_qry ="""
CREATE TABLE PR26 AS
WITH BENZO_RX AS
(
SELECT A.PRSCRB_PROV_LOC_ID, A.CLM_LNE_FACT_SK, A.MCAID_ID, A.PRSRB_DT, A.DRUG_DSPN_DT
FROM CS_MASTER A
WHERE A."Class" = 'Benzo'
)
,

SUB_RX AS
(
SELECT A.PRSCRB_PROV_LOC_ID, A.CLM_LNE_FACT_SK, A.MCAID_ID, A.PRSRB_DT, A.DRUG_DSPN_DT
FROM CS_MASTER A
WHERE A.GENNME IN ('Buprenorphine/naloxone')
)

SELECT M.PRSCRB_PROV_LOC_ID, X.CNT_MEM_SUB, COUNT(DISTINCT M.MCAID_ID) AS CNT_MEM_TOT, 
(X.CNT_MEM_SUB/COUNT(DISTINCT M.MCAID_ID)) AS PR26
--CASE WHEN COUNT(DISTINCT M.MCAID_ID) < 10 THEN (X.CNT_MEM_SUB/COUNT(DISTINCT M.MCAID_ID)) * (COUNT(DISTINCT M.MCAID_ID) / 10)
--                ELSE (X.CNT_MEM_SUB/COUNT(DISTINCT M.MCAID_ID))
--                END AS PR26



FROM CS_MASTER M
INNER JOIN
(SELECT A.PRSCRB_PROV_LOC_ID, COUNT(DISTINCT A.MCAID_ID) AS CNT_MEM_SUB
FROM BENZO_RX A
INNER JOIN SUB_RX B
ON A.MCAID_ID = B.MCAID_ID
AND A.DRUG_DSPN_DT BETWEEN (B.DRUG_DSPN_DT - 7) AND (B.DRUG_DSPN_DT +7)
GROUP BY A.PRSCRB_PROV_LOC_ID) X
ON M.PRSCRB_PROV_LOC_ID = X.PRSCRB_PROV_LOC_ID
GROUP BY M.PRSCRB_PROV_LOC_ID, X.CNT_MEM_SUB
""".format(date1 = begin_date, date2 = end_date)

engine.execute(p26_qry)

p26_df = pd.read_sql('select * from PR26 where rownum < 10', engine)

p26_df.head()

2020-10-28 23:27:33,618 INFO sqlalchemy.engine.base.Engine 
        BEGIN
           EXECUTE IMMEDIATE 'DROP TABLE ' || 'PR26';
        EXCEPTION
           WHEN OTHERS THEN
              IF SQLCODE != -942 THEN
                 RAISE;
              END IF;
        END;    
    
2020-10-28 23:27:33,620 INFO sqlalchemy.engine.base.Engine {}
2020-10-28 23:27:33,748 INFO sqlalchemy.engine.base.Engine 
CREATE TABLE PR26 AS
WITH BENZO_RX AS
(
SELECT A.PRSCRB_PROV_LOC_ID, A.CLM_LNE_FACT_SK, A.MCAID_ID, A.PRSRB_DT, A.DRUG_DSPN_DT
FROM CS_MASTER A
WHERE A."Class" = 'Benzo'
)
,

SUB_RX AS
(
SELECT A.PRSCRB_PROV_LOC_ID, A.CLM_LNE_FACT_SK, A.MCAID_ID, A.PRSRB_DT, A.DRUG_DSPN_DT
FROM CS_MASTER A
WHERE A.GENNME IN ('Buprenorphine/naloxone')
)

SELECT M.PRSCRB_PROV_LOC_ID, X.CNT_MEM_SUB, COUNT(DISTINCT M.MCAID_ID) AS CNT_MEM_TOT, 
(X.CNT_MEM_SUB/COUNT(DISTINCT M.MCAID_ID)) AS PR26
--CASE WHEN COUNT(DISTINCT M.MCAID_ID) < 10 THEN (X.CNT_MEM_SUB/COUNT(DISTINCT M.MCAID_ID)) * (COUNT(DISTINCT M.MCAID_ID

,prscrb_prov_loc_id,cnt_mem_sub,cnt_mem_tot,pr26
0,126479,6,42,0.142857
1,120939,2,63,0.031746
2,173605,1,107,0.009346
3,7168,3,182,0.016484
4,297,1,55,0.018182


In [40]:
drop_table_if_exists('PR27')

p27_qry ="""
CREATE TABLE PR27 AS
WITH OP_RX AS
(
SELECT A.PRSCRB_PROV_LOC_ID, A.CLM_LNE_FACT_SK, A.MCAID_ID, A.PRSRB_DT, A.DRUG_DSPN_DT
FROM CS_MASTER A
WHERE A."Class" = 'Opioid'
AND A.GENNME NOT IN ('Buprenorphine/naloxone')
)
,

NALOX_RX AS
(
SELECT A.PRSCRB_PROV_LOC_ID, A.CLM_LNE_FACT_SK, A.MCAID_ID, A.PRSRB_DT, A.DRUG_DSPN_DT
FROM CS_MASTER A
WHERE A.GENNME IN ('Naloxone Hydrochloride/pentazocine Hydrochloride')
)

SELECT M.PRSCRB_PROV_LOC_ID, X.CNT_MEM_NALOX, COUNT(DISTINCT M.MCAID_ID) AS CNT_MEM_TOT, 
(X.CNT_MEM_NALOX/COUNT(DISTINCT M.MCAID_ID)) AS PR27

--CASE WHEN COUNT(DISTINCT M.MCAID_ID) < 10 THEN (X.CNT_MEM_NALOX/COUNT(DISTINCT M.MCAID_ID)) * (COUNT(DISTINCT M.MCAID_ID) / 10)
--                ELSE (X.CNT_MEM_NALOX/COUNT(DISTINCT M.MCAID_ID))
--                END AS PR27
 


FROM CS_MASTER M
INNER JOIN
(SELECT A.PRSCRB_PROV_LOC_ID, COUNT(DISTINCT A.MCAID_ID) AS CNT_MEM_NALOX
FROM OP_RX A
INNER JOIN NALOX_RX B
ON A.MCAID_ID = B.MCAID_ID
AND A.DRUG_DSPN_DT BETWEEN (B.DRUG_DSPN_DT - 7) AND (B.DRUG_DSPN_DT +7)
GROUP BY A.PRSCRB_PROV_LOC_ID) X
ON M.PRSCRB_PROV_LOC_ID = X.PRSCRB_PROV_LOC_ID
GROUP BY M.PRSCRB_PROV_LOC_ID, X.CNT_MEM_NALOX
""".format(date1 = begin_date, date2 = end_date)

engine.execute(p27_qry)

p27_df = pd.read_sql('select * from PR27 where rownum < 10', engine)

p27_df.head()

2020-10-28 23:27:34,278 INFO sqlalchemy.engine.base.Engine 
        BEGIN
           EXECUTE IMMEDIATE 'DROP TABLE ' || 'PR27';
        EXCEPTION
           WHEN OTHERS THEN
              IF SQLCODE != -942 THEN
                 RAISE;
              END IF;
        END;    
    
2020-10-28 23:27:34,279 INFO sqlalchemy.engine.base.Engine {}
2020-10-28 23:27:34,384 INFO sqlalchemy.engine.base.Engine 
CREATE TABLE PR27 AS
WITH OP_RX AS
(
SELECT A.PRSCRB_PROV_LOC_ID, A.CLM_LNE_FACT_SK, A.MCAID_ID, A.PRSRB_DT, A.DRUG_DSPN_DT
FROM CS_MASTER A
WHERE A."Class" = 'Opioid'
AND A.GENNME NOT IN ('Buprenorphine/naloxone')
)
,

NALOX_RX AS
(
SELECT A.PRSCRB_PROV_LOC_ID, A.CLM_LNE_FACT_SK, A.MCAID_ID, A.PRSRB_DT, A.DRUG_DSPN_DT
FROM CS_MASTER A
WHERE A.GENNME IN ('Naloxone Hydrochloride/pentazocine Hydrochloride')
)

SELECT M.PRSCRB_PROV_LOC_ID, X.CNT_MEM_NALOX, COUNT(DISTINCT M.MCAID_ID) AS CNT_MEM_TOT, 
(X.CNT_MEM_NALOX/COUNT(DISTINCT M.MCAID_ID)) AS PR27

--CASE WHEN COUNT(DISTINCT M.MCAID_ID) < 1

,prscrb_prov_loc_id,cnt_mem_nalox,cnt_mem_tot,pr27
0,160380,2,95,0.021053
1,2755,3,228,0.013158
2,148485,1,18,0.055556
3,120451,1,7,0.142857
4,101951,1,17,0.058824


In [41]:
drop_table_if_exists('PR28')

p28_qry ="""
CREATE TABLE PR28 AS
WITH MEM_SUD_DIAG AS
(
SELECT DISTINCT A.MCAID_ID
FROM BIDM_USR_RPTS.CLM_LNE_FACT_V A
INNER JOIN CS_MASTER B
ON A.MCAID_ID = B.MCAID_ID
AND 
    (A.DIAG_1_CD LIKE 'F1%' OR 
    A.DIAG_2_CD LIKE 'F1%' OR 
    A.DIAG_3_CD LIKE 'F1%' OR 
    A.DIAG_4_CD LIKE 'F1%')
AND A.CURR_REC_IND = 'Y'
AND A.MOST_RCNT_CLM_IND = 'Y'
AND A.SRC_REC_DEL_IND = 'N'
AND A.CLM_STS_CD = 'P'
AND A.LNE_STS_CD = 'P'
AND A.RVRSL_IND = 'N'
AND A.ENC_IND = 'N'
AND A.CLM_PD_DT >= TO_DATE(\'{date1}\','MMDDYYYY')
AND A.LNE_FRST_SVC_DT BETWEEN TO_DATE(\'{date1}\','MMDDYYYY') and TO_DATE(\'{date2}\','MMDDYYYY')
)

SELECT M.PRSCRB_PROV_LOC_ID, SUD.CNT_MEM_SUD_DIAG, COUNT(DISTINCT M.MCAID_ID) AS CNT_MEM_TOT,
(SUD.CNT_MEM_SUD_DIAG/COUNT(DISTINCT M.MCAID_ID)) AS PR28

--CASE WHEN COUNT(DISTINCT M.MCAID_ID) < 10 THEN (SUD.CNT_MEM_SUD_DIAG/COUNT(DISTINCT M.MCAID_ID)) * (COUNT(DISTINCT M.MCAID_ID) / 10)
--                ELSE (SUD.CNT_MEM_SUD_DIAG/COUNT(DISTINCT M.MCAID_ID))
--                END AS PR28


FROM CS_MASTER M
INNER JOIN
(SELECT A.PRSCRB_PROV_LOC_ID, COUNT(DISTINCT A.MCAID_ID) AS CNT_MEM_SUD_DIAG
FROM CS_MASTER A
WHERE A.MCAID_ID IN (SELECT DISTINCT MCAID_ID FROM MEM_SUD_DIAG)
GROUP BY A.PRSCRB_PROV_LOC_ID) SUD
ON M.PRSCRB_PROV_LOC_ID = SUD.PRSCRB_PROV_LOC_ID
GROUP BY M.PRSCRB_PROV_LOC_ID, SUD.CNT_MEM_SUD_DIAG
""".format(date1 = begin_date, date2 = end_date)

engine.execute(p28_qry)

p28_df = pd.read_sql('select * from PR28 where rownum < 10', engine)

p28_df.head()

2020-10-28 23:27:34,873 INFO sqlalchemy.engine.base.Engine 
        BEGIN
           EXECUTE IMMEDIATE 'DROP TABLE ' || 'PR28';
        EXCEPTION
           WHEN OTHERS THEN
              IF SQLCODE != -942 THEN
                 RAISE;
              END IF;
        END;    
    
2020-10-28 23:27:34,875 INFO sqlalchemy.engine.base.Engine {}
2020-10-28 23:27:34,984 INFO sqlalchemy.engine.base.Engine 
CREATE TABLE PR28 AS
WITH MEM_SUD_DIAG AS
(
SELECT DISTINCT A.MCAID_ID
FROM BIDM_USR_RPTS.CLM_LNE_FACT_V A
INNER JOIN CS_MASTER B
ON A.MCAID_ID = B.MCAID_ID
AND 
    (A.DIAG_1_CD LIKE 'F1%' OR 
    A.DIAG_2_CD LIKE 'F1%' OR 
    A.DIAG_3_CD LIKE 'F1%' OR 
    A.DIAG_4_CD LIKE 'F1%')
AND A.CURR_REC_IND = 'Y'
AND A.MOST_RCNT_CLM_IND = 'Y'
AND A.SRC_REC_DEL_IND = 'N'
AND A.CLM_STS_CD = 'P'
AND A.LNE_STS_CD = 'P'
AND A.RVRSL_IND = 'N'
AND A.ENC_IND = 'N'
AND A.CLM_PD_DT >= TO_DATE('04012020','MMDDYYYY')
AND A.LNE_FRST_SVC_DT BETWEEN TO_DATE('04012020','MMDDYYYY') and TO_DATE('06302020','MMDDYYYY

,prscrb_prov_loc_id,cnt_mem_sud_diag,cnt_mem_tot,pr28
0,122721,28,126,0.222222
1,137727,6,14,0.428571
2,166203,4,27,0.148148
3,140831,5,22,0.227273
4,5298,1,8,0.125000


In [42]:
drop_table_if_exists('PR29')

p29_qry ="""
CREATE TABLE PR29 AS
WITH WEATHER AS
(
SELECT A.EVENT_ID, A.BEGIN_DATE, B.COUNTY, A.CZ_NAME_STR, B.NAME
FROM D98TOGLE.WEATHER_SEVERE A, D98TOGLE.NWS_ZONE_TO_COUNTY B -- TWO TABLES FROM THE NATIONAL WEATHER SERVICE.  COUNTY ISNT GREAT.  TABLES HAVE LAT/LONG CENTROID COORDS.  COULD RUN GEOG CLUSTERING TO MAKE IT MORE ACCURATE
WHERE A.CZ_FIPS = B."ZONE"
)
,
PROV_ADDR_CNTY AS
(
SELECT distinct A.CLM_LNE_FACT_SK, A.PRSCRB_PROV_LOC_ID, A.PRSRB_DT, A.LNE_FRST_SVC_DT, A.DRUG_DSPN_DT, B.PRSCRB_PROV_LOC_DIM_SK
from CS_MASTER A
INNER JOIN BIDM_USR_RPTS.CLM_LNE_FACT_V B
ON B.CLM_LNE_FACT_SK = A.CLM_LNE_FACT_SK
)
,
PROV_ADDR_CNTY_2 AS
(
SELECT A.*,B.PROV_LOC_ID, B.SVC_CNTY_DESC
FROM PROV_ADDR_CNTY A
INNER JOIN BIDM_USR_RPTS.PROV_LOC_DIM_V B
ON A.PRSCRB_PROV_LOC_DIM_SK = B.PROV_LOC_DIM_SK
)

SELECT M.PRSCRB_PROV_LOC_ID, X.CNT_WEATHER, 
(X.CNT_WEATHER/COUNT(DISTINCT M.CLM_LNE_FACT_SK)) AS PR29
--X.CNT_WEATHER as PR29

FROM CS_MASTER M
INNER JOIN
(SELECT C.PRSCRB_PROV_LOC_ID, count(distinct c.CLM_LNE_FACT_SK) AS CNT_WEATHER
FROM PROV_ADDR_CNTY_2 C
INNER JOIN WEATHER W
ON W.COUNTY = C.SVC_CNTY_DESC
AND (W.BEGIN_DATE = C.PRSRB_DT or
     W.BEGIN_DATE = C.LNE_FRST_SVC_DT or
     W.BEGIN_DATE = C.DRUG_DSPN_DT)
GROUP BY C.PRSCRB_PROV_LOC_ID
) X
ON M.PRSCRB_PROV_LOC_ID = X.PRSCRB_PROV_LOC_ID
GROUP BY M.PRSCRB_PROV_LOC_ID, X.CNT_WEATHER
""".format(date1 = begin_date, date2 = end_date)

engine.execute(p29_qry)

p29_df = pd.read_sql('select * from PR29 where rownum < 10', engine)

p29_df.head()

2020-10-28 23:28:09,217 INFO sqlalchemy.engine.base.Engine 
        BEGIN
           EXECUTE IMMEDIATE 'DROP TABLE ' || 'PR29';
        EXCEPTION
           WHEN OTHERS THEN
              IF SQLCODE != -942 THEN
                 RAISE;
              END IF;
        END;    
    
2020-10-28 23:28:09,219 INFO sqlalchemy.engine.base.Engine {}
2020-10-28 23:28:09,354 INFO sqlalchemy.engine.base.Engine 
CREATE TABLE PR29 AS
WITH WEATHER AS
(
SELECT A.EVENT_ID, A.BEGIN_DATE, B.COUNTY, A.CZ_NAME_STR, B.NAME
FROM D98TOGLE.WEATHER_SEVERE A, D98TOGLE.NWS_ZONE_TO_COUNTY B -- TWO TABLES FROM THE NATIONAL WEATHER SERVICE.  COUNTY ISNT GREAT.  TABLES HAVE LAT/LONG CENTROID COORDS.  COULD RUN GEOG CLUSTERING TO MAKE IT MORE ACCURATE
WHERE A.CZ_FIPS = B."ZONE"
)
,
PROV_ADDR_CNTY AS
(
SELECT distinct A.CLM_LNE_FACT_SK, A.PRSCRB_PROV_LOC_ID, A.PRSRB_DT, A.LNE_FRST_SVC_DT, A.DRUG_DSPN_DT, B.PRSCRB_PROV_LOC_DIM_SK
from CS_MASTER A
INNER JOIN BIDM_USR_RPTS.CLM_LNE_FACT_V B
ON B.CLM_LNE_FACT_SK = A.CLM_LNE_

,prscrb_prov_loc_id,cnt_weather,pr29
0,156748,1,0.058824
1,164840,10,0.057803
2,1716,2,0.013699
3,154364,2,0.009662
4,148121,4,0.036364


In [43]:
drop_table_if_exists('PR30')

p30_qry ="""
CREATE TABLE PR30 AS
SELECT X.PRSCRB_PROV_LOC_ID, Z.PR30 AS CNT_NAIVE, COUNT(DISTINCT X.MCAID_ID) AS CNT_MEM_TOT,
(Z.PR30/COUNT(DISTINCT X.MCAID_ID)) AS PR30

--CASE WHEN COUNT(DISTINCT X.MCAID_ID) < 10 THEN (Z.PR30/COUNT(DISTINCT X.MCAID_ID)) * (COUNT(DISTINCT X.MCAID_ID) / 10)
--                ELSE (Z.PR30/COUNT(DISTINCT X.MCAID_ID))
--                END AS PR30

FROM CS_MASTER X
INNER JOIN
(select prscrb_prov_loc_id,
count(distinct mcaid_id) as pr30
from
(select aaa.*, 
drug_dspn_dt - prev_drug_dspn_dt as "days_diff"
from
(select 
prscrb_prov_loc_id, 
mcaid_id, 
"LongShortActing", 
drug_dspn_dt,
CASE WHEN lower(clm.gennme) in ('morphine sulfate', 'methadone mydrochloride', 'fentanyl', 'oxymorphone hydrochloride') THEN 'ER' ELSE 'NOT_ER' END AS Extended_Release,
LAG (drug_dspn_dt,1) OVER (PARTITION BY prscrb_prov_loc_id, mcaid_id ORDER BY drug_dspn_dt) AS prev_drug_dspn_dt,
LAG ("MME_Conversion_Factor",1) OVER (PARTITION BY prscrb_prov_loc_id, mcaid_id ORDER BY drug_dspn_dt) AS prev_mme
from CS_MASTER clm
order by prscrb_prov_loc_id, mcaid_id, drug_dspn_dt) aaa) bbb
where 1=1
and bbb."days_diff" >=30
and bbb.extended_release = 'ER'
and bbb."LongShortActing" = 'LA'
and bbb.prev_mme > 0
group by prscrb_prov_loc_id) Z
ON X.PRSCRB_PROV_LOC_ID = Z.PRSCRB_PROV_LOC_ID
GROUP BY X.PRSCRB_PROV_LOC_ID, Z.PR30
""".format(date1 = begin_date, date2 = end_date)

engine.execute(p30_qry)

p30_df = pd.read_sql('select * from PR30 where rownum < 10', engine)

p30_df.head()

2020-10-28 23:28:40,686 INFO sqlalchemy.engine.base.Engine 
        BEGIN
           EXECUTE IMMEDIATE 'DROP TABLE ' || 'PR30';
        EXCEPTION
           WHEN OTHERS THEN
              IF SQLCODE != -942 THEN
                 RAISE;
              END IF;
        END;    
    
2020-10-28 23:28:40,688 INFO sqlalchemy.engine.base.Engine {}
2020-10-28 23:28:40,831 INFO sqlalchemy.engine.base.Engine 
CREATE TABLE PR30 AS
SELECT X.PRSCRB_PROV_LOC_ID, Z.PR30 AS CNT_NAIVE, COUNT(DISTINCT X.MCAID_ID) AS CNT_MEM_TOT,
(Z.PR30/COUNT(DISTINCT X.MCAID_ID)) AS PR30

--CASE WHEN COUNT(DISTINCT X.MCAID_ID) < 10 THEN (Z.PR30/COUNT(DISTINCT X.MCAID_ID)) * (COUNT(DISTINCT X.MCAID_ID) / 10)
--                ELSE (Z.PR30/COUNT(DISTINCT X.MCAID_ID))
--                END AS PR30

FROM CS_MASTER X
INNER JOIN
(select prscrb_prov_loc_id,
count(distinct mcaid_id) as pr30
from
(select aaa.*, 
drug_dspn_dt - prev_drug_dspn_dt as "days_diff"
from
(select 
prscrb_prov_loc_id, 
mcaid_id, 
"LongShortActing", 
drug

,prscrb_prov_loc_id,cnt_naive,cnt_mem_tot,pr30
0,160253,1,16,0.062500
1,179449,1,163,0.006135
2,114542,1,21,0.047619
3,121762,2,46,0.043478
4,129077,1,20,0.050000


# Put queries together

In [44]:
drop_table_if_exists('OP_ANALYSIS')

op_qry ="""
CREATE TABLE OP_ANALYSIS AS
SELECT A.*
, COALESCE(PR5.PR5, 0) AS PR5
, COALESCE(PR6.PR6, 0) AS PR6
,COALESCE(PR7.PR7, 0) AS PR7
,COALESCE(PR8.PR8, 0) AS PR8
,COALESCE(PR9.PR9, 0) AS PR9
,COALESCE(PR10.PR10, 0) AS PR10
,COALESCE(PR11.PR11, 0) AS PR11
--,PR12.PR12 skip
,COALESCE(PR13.PR13, 0) AS PR13
,COALESCE(PR15.CNT_CLM, 0) AS PR15
--,PR17.PR17
,COALESCE(PR18.PR18, 0) AS PR18
,COALESCE(PR19.PR19, 0) AS PR19
--,PR20.PR20 skip
,COALESCE(PR21.PR21, 0) AS PR21
,COALESCE(PR22.PR22, 0) AS PR22
,COALESCE(PR23.PR23, 0) AS PR23
,COALESCE(PR24.PR24, 0) AS PR24
,COALESCE(PR25.PR25, 0) AS PR25
,COALESCE(PR26.PR26, 0) AS PR26
,COALESCE(PR27.PR27, 0) AS PR27
,COALESCE(PR28.PR28, 0) AS PR28
,COALESCE(PR29.PR29, 0) AS PR29
,COALESCE(PR30.PR30, 0) AS PR30
FROM CS_MASTER_SUM A
LEFT JOIN PR5 ON PR5.PRSCRB_PROV_LOC_ID = A.PRSCRB_PROV_LOC_ID
LEFT JOIN PR6 ON PR6.PRSCRB_PROV_LOC_ID = A.PRSCRB_PROV_LOC_ID
LEFT JOIN PR7 ON PR7.PRSCRB_PROV_LOC_ID = A.PRSCRB_PROV_LOC_ID
LEFT JOIN PR8 ON PR8.PRSCRB_PROV_LOC_ID = A.PRSCRB_PROV_LOC_ID
LEFT JOIN PR9 ON PR9.PRSCRB_PROV_LOC_ID = A.PRSCRB_PROV_LOC_ID
LEFT JOIN PR10 ON PR10.OP_PRSCRB_PROV_LOC_ID = A.PRSCRB_PROV_LOC_ID
LEFT JOIN PR11 ON PR11.PRSCRB_PROV_LOC_ID = A.PRSCRB_PROV_LOC_ID
LEFT JOIN PR13 ON PR13.PRSCRB_PROV_LOC_ID = A.PRSCRB_PROV_LOC_ID
LEFT JOIN PR15 ON PR15.PRSCRB_PROV_LOC_ID = A.PRSCRB_PROV_LOC_ID
LEFT JOIN PR18 ON PR18.PRSCRB_PROV_LOC_ID = A.PRSCRB_PROV_LOC_ID
LEFT JOIN PR19 ON PR19.PRSCRB_PROV_LOC_ID = A.PRSCRB_PROV_LOC_ID
LEFT JOIN PR21 ON PR21.PRSCRB_PROV_LOC_ID = A.PRSCRB_PROV_LOC_ID
LEFT JOIN PR22 ON PR22.PRSCRB_PROV_LOC_ID = A.PRSCRB_PROV_LOC_ID
LEFT JOIN PR23 ON PR23.PRSCRB_PROV_LOC_ID = A.PRSCRB_PROV_LOC_ID
LEFT JOIN PR24 ON PR24.PRSCRB_PROV_LOC_ID = A.PRSCRB_PROV_LOC_ID
LEFT JOIN PR25 ON PR25.PRSCRB_PROV_LOC_ID = A.PRSCRB_PROV_LOC_ID
LEFT JOIN PR26 ON PR26.PRSCRB_PROV_LOC_ID = A.PRSCRB_PROV_LOC_ID
LEFT JOIN PR27 ON PR27.PRSCRB_PROV_LOC_ID = A.PRSCRB_PROV_LOC_ID
LEFT JOIN PR28 ON PR28.PRSCRB_PROV_LOC_ID = A.PRSCRB_PROV_LOC_ID
LEFT JOIN PR29 ON PR29.PRSCRB_PROV_LOC_ID = A.PRSCRB_PROV_LOC_ID
LEFT JOIN PR30 ON PR30.PRSCRB_PROV_LOC_ID = A.PRSCRB_PROV_LOC_ID
""".format(date1 = begin_date, date2 = end_date)

engine.execute(op_qry)

op_df = pd.read_sql('select * from OP_ANALYSIS where rownum < 10', engine)

op_df.head()

2020-10-28 23:28:43,409 INFO sqlalchemy.engine.base.Engine 
        BEGIN
           EXECUTE IMMEDIATE 'DROP TABLE ' || 'OP_ANALYSIS';
        EXCEPTION
           WHEN OTHERS THEN
              IF SQLCODE != -942 THEN
                 RAISE;
              END IF;
        END;    
    
2020-10-28 23:28:43,410 INFO sqlalchemy.engine.base.Engine {}
2020-10-28 23:28:43,519 INFO sqlalchemy.engine.base.Engine 
CREATE TABLE OP_ANALYSIS AS
SELECT A.*
, COALESCE(PR5.PR5, 0) AS PR5
, COALESCE(PR6.PR6, 0) AS PR6
,COALESCE(PR7.PR7, 0) AS PR7
,COALESCE(PR8.PR8, 0) AS PR8
,COALESCE(PR9.PR9, 0) AS PR9
,COALESCE(PR10.PR10, 0) AS PR10
,COALESCE(PR11.PR11, 0) AS PR11
--,PR12.PR12 skip
,COALESCE(PR13.PR13, 0) AS PR13
,COALESCE(PR15.CNT_CLM, 0) AS PR15
--,PR17.PR17
,COALESCE(PR18.PR18, 0) AS PR18
,COALESCE(PR19.PR19, 0) AS PR19
--,PR20.PR20 skip
,COALESCE(PR21.PR21, 0) AS PR21
,COALESCE(PR22.PR22, 0) AS PR22
,COALESCE(PR23.PR23, 0) AS PR23
,COALESCE(PR24.PR24, 0) AS PR24
,COALESCE(PR25.PR25, 0) AS PR25
,

,Member Count,Count Script,Sum Dispensed Quantity,Max Dispense Date,Min Dispense Date,prscrb_prov_loc_id,Total Days,pr2,pr1,pr4,pr3,pr5,pr6,pr7,pr8,pr9,pr10,pr11,pr13,pr15,pr18,pr19,pr21,pr22,pr23,pr24,pr25,pr26,pr27,pr28,pr29,pr30
0,32,80,4119,2020-06-29,2020-04-01,105691,90,2.5,128.718750,0.888889,45.766667,0.060000,0.03125,5,0,0,0,0.1125,0.32,0,0,0,0,6.439075,0,0,0,0.03125,0,0.21875,0,0.03125
1,4,4,266,2020-06-19,2020-04-29,105693,52,1.0,66.500000,0.076923,5.115385,0.061224,0.00000,0,0,0,0,0.2500,0.00,0,0,0,0,1.758067,0,0,0,0.00000,0,0.00000,0,0.00000
2,1,1,30,2020-04-30,2020-04-30,105699,1,1.0,30.000000,1.000000,30.000000,0.000000,0.00000,0,0,0,0,0.0000,0.00,0,0,0,0,13.271845,0,0,0,0.00000,0,0.00000,0,0.00000
3,5,9,215,2020-06-10,2020-05-18,105700,24,1.8,43.000000,0.375000,8.958333,0.108696,0.00000,3,0,0,0,1.0000,0.00,0,0,0,0,6.675857,0,0,0,0.00000,0,0.40000,0,0.00000
4,3,6,340,2020-06-29,2020-04-04,105707,87,2.0,113.333333,0.068966,3.908046,0.032258,0.00000,0,0,0,0,0.0000,0.00,0,0,0,0,17.707574,0,0,0,0.00000,0,0.00000,0,0.00000


In [45]:
# if save_csv:
# @hidden_cell
# This connection object is used to access your data and contains your credentials.
# You might want to remove those credentials before you share your notebook.

from project_lib import Project
project = Project.access()
OP_ANALYSIS_credentials = project.get_connected_data(name="OP_ANALYSIS")

import pandas as pd, cx_Oracle

OP_ANALYSIS_dsn = cx_Oracle.makedsn(host = OP_ANALYSIS_credentials['host'], port = OP_ANALYSIS_credentials['port'], service_name = OP_ANALYSIS_credentials['service_name'])
OP_ANALYSIS_connection = cx_Oracle.connect(OP_ANALYSIS_credentials['username'], OP_ANALYSIS_credentials['password'], OP_ANALYSIS_dsn, encoding = 'UTF-8', nencoding = 'UTF-8')

query = 'SELECT * FROM OP_ANALYSIS'
cursor = OP_ANALYSIS_connection.cursor()

cursor.execute(query)
query_data = cursor.fetchall()

col_names = [colname[0] for colname in cursor.description]
data_df_1 = pd.DataFrame(data=query_data, columns=col_names)
data_df_1.head()

,Member Count,Count Script,Sum Dispensed Quantity,Max Dispense Date,Min Dispense Date,PRSCRB_PROV_LOC_ID,Total Days,PR2,PR1,PR4,PR3,PR5,PR6,PR7,PR8,PR9,PR10,PR11,PR13,PR15,PR18,PR19,PR21,PR22,PR23,PR24,PR25,PR26,PR27,PR28,PR29,PR30
0,32,80,4119.0,2020-06-29,2020-04-01,105691,90,2.5,128.718750,0.888889,45.766667,0.060000,0.03125,5,0.0,0.0,0.0,0.1125,0.32,0,0,0.0,0.0,6.439075,0.0,0.0,0.0,0.03125,0.0,0.21875,0.0,0.03125
1,4,4,266.0,2020-06-19,2020-04-29,105693,52,1.0,66.500000,0.076923,5.115385,0.061224,0.00000,0,0.0,0.0,0.0,0.2500,0.00,0,0,0.0,0.0,1.758067,0.0,0.0,0.0,0.00000,0.0,0.00000,0.0,0.00000
2,1,1,30.0,2020-04-30,2020-04-30,105699,1,1.0,30.000000,1.000000,30.000000,0.000000,0.00000,0,0.0,0.0,0.0,0.0000,0.00,0,0,0.0,0.0,13.271845,0.0,0.0,0.0,0.00000,0.0,0.00000,0.0,0.00000
3,5,9,215.0,2020-06-10,2020-05-18,105700,24,1.8,43.000000,0.375000,8.958333,0.108696,0.00000,3,0.0,0.0,0.0,1.0000,0.00,0,0,0.0,0.0,6.675857,0.0,0.0,0.0,0.00000,0.0,0.40000,0.0,0.00000
4,3,6,340.0,2020-06-29,2020-04-04,105707,87,2.0,113.333333,0.068966,3.908046,0.032258,0.00000,0,0.0,0.0,0.0,0.0000,0.00,0,0,0.0,0.0,17.707574,0.0,0.0,0.0,0.00000,0.0,0.00000,0.0,0.00000


# Save dataset back to assets folder

In [46]:
if save_csv:
    filename = file_prefix + '_' + re.sub("'","",begin_date) + '_' + re.sub("'","",end_date) + '.csv'
    project.save_data(filename, data_df_1.to_csv(index=False), overwrite=True)

## Scratch work

In [47]:
# pd.read_sql('select * from CS_MASTER where rownum<10', engine)

In [48]:
# qqq = """
# select mcaid_id, "Class", drug_dspn_dt
# --LISTAGG("Class", ',') WITHIN GROUP (ORDER BY drug_dspn_dt) AS drugs,
# --LISTAGG(drug_dspn_dt, ',') WITHIN GROUP (ORDER BY drug_dspn_dt) AS dates
# from CS_MASTER
# where 1=1
# and "Class" in ('Opioid', 'Benzo', 'Muscle Relaxant')
# order by mcaid_id, drug_dspn_dt
# """

# # Setup mme per day array
# temp = pd.read_sql(qqq, engine)

# temp.head()

In [49]:
# min_dates = temp.groupby(['mcaid_id'], as_index = False)['drug_dspn_dt'].min().rename(columns={"drug_dspn_dt":"min_drug_dspn_dt"})
# temp2 = temp.merge(min_dates, on = 'mcaid_id', how = 'left')
# temp2['index'] = temp2['drug_dspn_dt'] - temp2['min_drug_dspn_dt']

# temp2.head()

In [50]:
# import numpy as np
# # # items = np.random.choice(['A','B','C'], 100)
# # # index = np.random.choice(np.arange(50), 100)
# # # df = pd.DataFrame({'items':items,'index':index})

# # df = temp2

# # df.head(5)

# temp2

In [51]:
# all_res = []
# all_ids = list(temp2.mcaid_id.drop_duplicates())

# for iid in all_ids:
#     df = temp2[temp2.mcaid_id==iid].reset_index(drop = True)
#     data = df.sort_values(['index','Class']).values
#     n = df.shape[0]
#     m,p,q = 0,0,0
#     reset = False
#     result = []
#     counts = 0
#     for i in range(m,n):
#         if reset: reset = False 
#         if data[i,1] != 'Opioid': continue
#         for j in range(p,n):
#             if reset: break
#             if data[j,1] != 'Benzo': continue
#             p = j
#             for k in range(q,n):
#                 counts += 1
#                 if data[k,1] != 'Muscle Relaxant': continue
#                 # check data
#                 q=k
#                 low = min(data[i,4].days, data[j,4].days, data[k,4].days)
#                 high = max(data[i,4].days, data[j,4].days, data[k,4].days)
#                 if high - low < 30:
#                     result.append((i,j,k))
#                 else:
#                     pivot = np.argmin([data[i,4].days, data[j,4].days, data[k,4].days])
#                     if pivot == 2:
#                         continue
#                     elif pivot == 1:
#                         break
#                     else:
#                         reset = True
#                         break   
#     all_res.append(result)
    

    
    
    
    
    
    
    
    
    
    
# # print(result)